<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/MVQAN_VQC_STEP2D_BLOCKWISE_REUPLOAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVQAN VQC Step 2D — Block-Specific Semantic Data Re-uploading

This notebook tests one controlled, VQC-centered change after Step 2C:

- preserve the six semantic EEG view groups;
- keep the compact six-qubit, quantum-measurement-only classifier;
- generate a different `RY/RZ` angle pair for every semantic view **for every quantum block**;
- re-upload those block-specific summaries inside the VQC.

The Step-2C model is retained as the matched ablation. There is no classical prediction
anchor, probability fusion, residual bypass, pretrained classical classifier, or direct
projector-to-output path.


# 1. Install dependencies


In [1]:
!pip -q install moabb mne pyriemann pennylane scikit-learn pandas numpy scipy tqdm kaggle kagglehub h5py mat73 requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.7/837.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.

# 2. Imports and focused configuration


In [2]:
import os, gc, math, random, warnings, time, json, re

from dataclasses import dataclass, asdict

from typing import Dict, List, Tuple, Optional

import numpy as np

import pandas as pd

import scipy.signal as sps

from scipy.linalg import eigh

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (
    f1_score, balanced_accuracy_score, roc_auc_score, average_precision_score,
    accuracy_score, confusion_matrix, precision_score, recall_score
)

from sklearn.decomposition import PCA

from sklearn.utils.class_weight import compute_class_weight

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import pennylane as qml

warnings.filterwarnings("ignore")

@dataclass

class CFG:
    # ---------------- RUN CONTROL ----------------
    RUN_MODE: str = "quick"  # quick -> tune -> paper
    SEED: int = 42
    ACTIVE_DATASETS: Tuple[str, ...] = ("BNCI-P300", "UCL-ALS-EEG")

    # ---------------- DATASET ----------------
    DATASET_CANDIDATES: Tuple[str, ...] = (
        "BNCI2014_009", "BNCI2014_008", "bi2013a", "BI2013a", "bi2012", "BrainInvaders2013a"
    )
    FMIN: float = 0.1
    FMAX: float = 30.0
    TMIN: float = 0.0
    TMAX: float = 0.8
    RESAMPLE: int = 128

    MAX_SUBJECTS_QUICK: int = 6
    MAX_SUBJECTS_TUNE: Optional[int] = 12
    MAX_SUBJECTS_PAPER: Optional[int] = None
    MAX_OUTER_FOLDS_QUICK: int = 3
    MAX_OUTER_FOLDS_TUNE: int = 5
    MAX_OUTER_FOLDS_PAPER: Optional[int] = None

    # ---------------- COMMON TRAINING ----------------
    EPOCHS_QUICK: int = 12
    EPOCHS_TUNE: int = 30
    EPOCHS_PAPER: int = 50
    PATIENCE_QUICK: int = 4
    PATIENCE_TUNE: int = 8
    PATIENCE_PAPER: int = 12
    BATCH_SIZE_CPU: int = 32
    BATCH_SIZE_GPU: int = 128
    AUTO_BATCH_BY_DEVICE: bool = True
    TORCH_NUM_THREADS_CPU: int = 4
    FORCE_QML_CPU: bool = True
    QML_BATCH_SIZE_CPU: int = 32
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    DROPOUT: float = 0.20
    USE_BALANCED_SAMPLER: bool = True
    LOSS_TYPE: str = "focal"
    FOCAL_GAMMA: float = 2.0
    TARGET_CLASS_WEIGHT_MULTIPLIER: float = 1.25
    TUNE_THRESHOLD: bool = True

    # ---------------- FOCUSED VQC EXPERIMENT ----------------
    RUN_SKLEARN_REFERENCES: bool = True
    RUN_EEGNET_REFERENCE: bool = True
    RUN_LEGACY_VQC: bool = False
    RUN_FULLVIEW_LAYERWISE_VQC: bool = False  # historical Step-2B; not rerun in focused Step-2D
    RUN_SEMANTIC_QUBIT_VQC: bool = True
    RUN_SEMANTIC_REUPLOAD_VQC: bool = True

    N_QUBITS: int = 4
    LEGACY_VQC_DEPTH: int = 1
    FULLVIEW_CLIP_Z: float = 6.0
    MIN_FULLVIEW_INPUT_DIM: int = 8
    SEMANTIC_GROUP_ORDER: Tuple[str, ...] = ("temporal", "spatial", "frequency", "statistical", "covariance", "supervised")
    SEMANTIC_PROJECTOR_HIDDEN: int = 2
    SEMANTIC_PROJECTOR_DROPOUT: float = 0.05
    SEMANTIC_REUPLOAD_HIDDEN: int = 2
    PROJECTOR_HIDDEN: int = 32
    PROJECTOR_DROPOUT: float = 0.10
    PROJECTED_VQC_BLOCKS: int = 2
    Q_INIT_STD: float = 0.02
    Q_GRAD_CLIP: float = 1.0

    # Phase epochs: block-1, block-2, joint fine-tuning.
    Q_PHASE_EPOCHS_QUICK: Tuple[int, int, int] = (10, 6, 8)
    Q_PHASE_EPOCHS_TUNE: Tuple[int, int, int] = (18, 10, 17)
    Q_PHASE_EPOCHS_PAPER: Tuple[int, int, int] = (25, 15, 30)
    Q_PHASE_PATIENCE_QUICK: int = 5
    Q_PHASE_PATIENCE_TUNE: int = 8
    Q_PHASE_PATIENCE_PAPER: int = 10

    PROJECTION_LR: float = 5e-4
    QUANTUM_LR: float = 2e-3
    READOUT_LR: float = 1e-3
    Q_WEIGHT_DECAY: float = 1e-5

    # ---------------- VIEW/FOLD PREPARATION ----------------
    ENABLE_VIEW_PCA_CAP: bool = True
    VIEW_PCA_DIM_QUICK: int = 16
    VIEW_PCA_DIM_TUNE: int = 24
    VIEW_PCA_DIM_PAPER: int = 32
    ADD_SUPERVISED_SPATIAL_VIEW: bool = True
    SUPERVISED_VIEW_PCA_DIM: int = 16
    USE_BALANCED_SUBJECT_FOLDS: bool = True
    STRICT_BINARY_TEST_FOLDS: bool = True
    MIN_TEST_PER_CLASS: int = 5
    MIN_VAL_PER_CLASS: int = 2
    BALANCE_TRAIN_INDICES_FOR_TASKS: Tuple[str, ...] = ("als_control", "p300")
    MAX_TRAIN_CLASS_RATIO: float = 1.0

    RESULTS_DIR: str = "mvqan_vqc_step2d_results"
    STEP2C_REFERENCE: str = "SemanticQubit_Layerwise_VQC"
    MAIN_CANDIDATE: str = "SemanticReupload_Layerwise_VQC"

cfg = CFG()

if not torch.cuda.is_available():
    try:
        torch.set_num_threads(cfg.TORCH_NUM_THREADS_CPU)
    except Exception:
        pass


In [3]:
if cfg.RUN_MODE == "quick":
    cfg.MAX_SUBJECTS = cfg.MAX_SUBJECTS_QUICK
    cfg.MAX_OUTER_FOLDS = cfg.MAX_OUTER_FOLDS_QUICK
    cfg.EPOCHS = cfg.EPOCHS_QUICK
    cfg.PATIENCE = cfg.PATIENCE_QUICK
    cfg.VIEW_PCA_DIM = cfg.VIEW_PCA_DIM_QUICK
    cfg.Q_PHASE_EPOCHS = cfg.Q_PHASE_EPOCHS_QUICK
    cfg.Q_PHASE_PATIENCE = cfg.Q_PHASE_PATIENCE_QUICK
    cfg.SEEDS = (cfg.SEED,)
elif cfg.RUN_MODE == "tune":
    cfg.MAX_SUBJECTS = cfg.MAX_SUBJECTS_TUNE
    cfg.MAX_OUTER_FOLDS = cfg.MAX_OUTER_FOLDS_TUNE
    cfg.EPOCHS = cfg.EPOCHS_TUNE
    cfg.PATIENCE = cfg.PATIENCE_TUNE
    cfg.VIEW_PCA_DIM = cfg.VIEW_PCA_DIM_TUNE
    cfg.Q_PHASE_EPOCHS = cfg.Q_PHASE_EPOCHS_TUNE
    cfg.Q_PHASE_PATIENCE = cfg.Q_PHASE_PATIENCE_TUNE
    cfg.SEEDS = (11, 22)
else:
    cfg.MAX_SUBJECTS = cfg.MAX_SUBJECTS_PAPER
    cfg.MAX_OUTER_FOLDS = cfg.MAX_OUTER_FOLDS_PAPER
    cfg.EPOCHS = cfg.EPOCHS_PAPER
    cfg.PATIENCE = cfg.PATIENCE_PAPER
    cfg.VIEW_PCA_DIM = cfg.VIEW_PCA_DIM_PAPER
    cfg.Q_PHASE_EPOCHS = cfg.Q_PHASE_EPOCHS_PAPER
    cfg.Q_PHASE_PATIENCE = cfg.Q_PHASE_PATIENCE_PAPER
    cfg.SEEDS = (11, 22, 33)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg.BATCH_SIZE = cfg.BATCH_SIZE_GPU if (cfg.AUTO_BATCH_BY_DEVICE and DEVICE == "cuda") else cfg.BATCH_SIZE_CPU

os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

print("Global PyTorch device:", DEVICE)

print("Quantum models forced to CPU:", cfg.FORCE_QML_CPU)

print("Active datasets:", cfg.ACTIVE_DATASETS)

print("Q phase epochs:", cfg.Q_PHASE_EPOCHS)

print(json.dumps(asdict(cfg), indent=2, default=str))


Global PyTorch device: cpu
Quantum models forced to CPU: True
Active datasets: ('BNCI-P300', 'UCL-ALS-EEG')
Q phase epochs: (10, 6, 8)
{
  "RUN_MODE": "quick",
  "SEED": 42,
  "ACTIVE_DATASETS": [
    "BNCI-P300",
    "UCL-ALS-EEG"
  ],
  "DATASET_CANDIDATES": [
    "BNCI2014_009",
    "BNCI2014_008",
    "bi2013a",
    "BI2013a",
    "bi2012",
    "BrainInvaders2013a"
  ],
  "FMIN": 0.1,
  "FMAX": 30.0,
  "TMIN": 0.0,
  "TMAX": 0.8,
  "RESAMPLE": 128,
  "MAX_SUBJECTS_QUICK": 6,
  "MAX_SUBJECTS_TUNE": 12,
  "MAX_SUBJECTS_PAPER": null,
  "MAX_OUTER_FOLDS_QUICK": 3,
  "MAX_OUTER_FOLDS_TUNE": 5,
  "MAX_OUTER_FOLDS_PAPER": null,
  "EPOCHS_QUICK": 12,
  "EPOCHS_TUNE": 30,
  "EPOCHS_PAPER": 50,
  "PATIENCE_QUICK": 4,
  "PATIENCE_TUNE": 8,
  "PATIENCE_PAPER": 12,
  "BATCH_SIZE_CPU": 32,
  "BATCH_SIZE_GPU": 128,
  "AUTO_BATCH_BY_DEVICE": true,
  "TORCH_NUM_THREADS_CPU": 4,
  "FORCE_QML_CPU": true,
  "QML_BATCH_SIZE_CPU": 32,
  "LR": 0.001,
  "WEIGHT_DECAY": 0.0001,
  "DROPOUT": 0.2,
  "USE_BAL

# 3. Exact dataset registry


In [4]:
# ---------------- Exact advisor dataset mode ----------------
# This notebook uses the exact three datasets requested by the advisor.
DATASET_MODE = "advisor_exact"
REQUIRE_ALL_ADVISOR_DATASETS = (cfg.RUN_MODE == "paper")
ALLOW_NON_ALS_FALLBACK = False

# UCL/Figshare article information.
# Human-readable page:
# https://rdr.ucl.ac.uk/articles/dataset/Longitudinal_ALS_EEG_Dataset_for_Motor_Imagery_Studies/28156016
UCL_ALS_FIGSHARE_ARTICLE_ID = "28156016"
UCL_ALS_ARTICLE_PAGE = "https://rdr.ucl.ac.uk/articles/dataset/Longitudinal_ALS_EEG_Dataset_for_Motor_Imagery_Studies/28156016"
UCL_ALS_LOCAL_PATH = ""  # optional: point to an uploaded/extracted folder containing the eight .mat files
UCL_ALS_FIGSHARE_FILE_IDS = (
    51526682, 51526688, 51526691, 51526694,
    51526697, 51526700, 51526706, 51526709,
)

# Kaggle slug from the provided URL:
# https://www.kaggle.com/datasets/patrickiitmz/eeget-als-dataset
EEGET_ALS_KAGGLE_SLUG = "patrickiitmz/eeget-als-dataset"

# EEGET has a nested subject/time/scenario folder structure. The fixed loader infers subject and label
# from the folder path and uses EEG-like CSVs first. If no EEG-like CSV is found, it can use ET.csv as
# a runnable fallback, but this fallback must be reported as eye-tracking/multimodal, not EEG-only.
EEGET_ALLOW_ET_FALLBACK = True      # set False for strict EEG-only final run
EEGET_WINDOW_SECONDS = 2.0
EEGET_STRIDE_SECONDS = 2.0
EEGET_MAX_WINDOWS_PER_SUBJECT = 160 # keeps runtime manageable; fold-level balancing fixes class imbalance

ADVISOR_DATASETS = [
    {
        "name": "BNCI-P300",
        "task_type": "p300",
        "source_type": "moabb",
        "moabb_candidates": (
            "BNCI2014_009", "BNCI2014_008", "BNCI2015_003", "EPFLP300",
            "BI2012", "BI2013a", "BI2013b", "BrainInvaders2013"
        ),
        "include_keywords": ("bnci", "p300", "epfl"),
        "description": "Advisor-prioritized BNCI/clinical P300 dataset loaded through MOABB when available.",
    },
    {
        "name": "UCL-ALS-EEG",
        "task_type": "mi",
        "source_type": "ucl_figshare_als_mi",
        "figshare_article_id": UCL_ALS_FIGSHARE_ARTICLE_ID,
        "article_page": UCL_ALS_ARTICLE_PAGE,
        "local_path": UCL_ALS_LOCAL_PATH,
        "figshare_file_ids": UCL_ALS_FIGSHARE_FILE_IDS,
        "mi_task": "left_right",  # L vs R; excludes Re/rest for binary MI classification
        "sfreq": 128,             # fallback only; exact metadata is used if the MAT exposes it
        "subject_grouping": "file_patient",  # critical: group all L/R trials from one patient file as one subject
        "description": "Longitudinal ALS EEG motor-imagery dataset; left-vs-right MI binary task by default.",
    },
    {
        "name": "EEGET-ALS",
        "task_type": "als_control",  # path-inferred ALS vs non-ALS subject classification
        "source_type": "kaggle_or_url_or_local",
        "kaggle_slug": EEGET_ALS_KAGGLE_SLUG,
        "direct_url": "",
        "local_path": "",
        "auto_make_binary": True,
        "allow_et_fallback": EEGET_ALLOW_ET_FALLBACK,
        "window_seconds": EEGET_WINDOW_SECONDS,
        "stride_seconds": EEGET_STRIDE_SECONDS,
        "max_windows_per_subject": EEGET_MAX_WINDOWS_PER_SUBJECT,
        "sfreq": 128,
        "description": "Kaggle EEGET-ALS. Loader infers subject/ALS-control label from folders and windows EEG-like CSVs; ET fallback is explicit if used.",
    },
]

print(f"DATASET_MODE = {DATASET_MODE}")
print(f"Strict dataset loading = {REQUIRE_ALL_ADVISOR_DATASETS} (enabled only in paper mode)")
print("Datasets scheduled for loading:")
for d in ADVISOR_DATASETS:
    print(" -", d["name"], "|", d["source_type"], "|", d.get("description", ""))


DATASET_MODE = advisor_exact
Strict dataset loading = False (enabled only in paper mode)
Datasets scheduled for loading:
 - BNCI-P300 | moabb | Advisor-prioritized BNCI/clinical P300 dataset loaded through MOABB when available.
 - UCL-ALS-EEG | ucl_figshare_als_mi | Longitudinal ALS EEG motor-imagery dataset; left-vs-right MI binary task by default.
 - EEGET-ALS | kaggle_or_url_or_local | Kaggle EEGET-ALS. Loader infers subject/ALS-control label from folders and windows EEG-like CSVs; ET fallback is explicit if used.


# 4. Reproducibility


In [5]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(cfg.SEED)

## 4. Dataset utilities


In [6]:
def label_to_binary(labels):
    """Robustly map P300/ERP labels to 0/1. Positive class is Target/P300 if detectable; otherwise minority class."""
    labels_arr = np.asarray(labels)
    labels_str = labels_arr.astype(str)
    low = np.char.lower(labels_str)
    is_target = np.array([
        ("target" in s and "non" not in s) or (s in ["1", "true", "p300", "positive", "target"])
        for s in low
    ])
    if is_target.sum() > 0 and is_target.sum() < len(is_target):
        return is_target.astype(np.int64)
    vals, counts = np.unique(labels_str, return_counts=True)
    pos_val = vals[np.argmin(counts)]
    print(f"[label_to_binary] Could not infer Target/P300. Using minority label as positive: {pos_val}")
    return (labels_str == pos_val).astype(np.int64)

def normalize_label_ids(y):
    """Map arbitrary labels to 0..K-1."""
    vals = sorted(np.unique(y).tolist())
    mapping = {v: i for i, v in enumerate(vals)}
    return np.asarray([mapping[v] for v in y], dtype=np.int64), mapping

def ensure_trial_tensor(X):
    """Ensure EEG/features tensor has shape [N, C, T]."""
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 2:
        X = X[:, None, :]  # feature-table fallback: channels=1, time/features=T
    if X.ndim != 3:
        raise ValueError(f"Expected X with 2 or 3 dims; got shape {X.shape}")
    return X

USED_MOABB_CLASSES = set()

def _dataset_matches_keywords(name, ds, include_keywords=None):
    """Heuristic scanner for public-auto mode only. It helps select different public P300 datasets."""
    if not include_keywords:
        return True
    text_parts = [name]
    for attr in ["code", "paradigm", "doi", "event_id"]:
        try:
            text_parts.append(str(getattr(ds, attr, "")))
        except Exception:
            pass
    text = " ".join(text_parts).lower()
    return any(k.lower() in text for k in include_keywords)

def _is_p300_like_dataset(ds):
    event_id = getattr(ds, "event_id", {}) or {}
    keys = " ".join([str(k).lower() for k in event_id.keys()])
    vals = " ".join([str(v).lower() for v in event_id.values()])
    paradigm = str(getattr(ds, "paradigm", "")).lower()
    return (("target" in keys and "non" in keys) or ("p300" in keys) or ("p300" in paradigm) or ("target" in vals))


### 4.1 MOABB P300 loader


In [7]:
def instantiate_moabb_dataset(spec):
    """Instantiate a MOABB dataset from explicit candidates; in public_auto mode scan for P300-like alternatives.

    This function avoids reusing the same MOABB class for multiple public-auto datasets.
    """
    import inspect
    import moabb.datasets as mds
    candidates = spec.get("moabb_candidates", cfg.DATASET_CANDIDATES)
    include_keywords = spec.get("include_keywords", None)
    last_error = None

    # 1) Try explicit candidates in order.
    for name in candidates:
        if name in USED_MOABB_CLASSES:
            continue
        if hasattr(mds, name):
            cls = getattr(mds, name)
            try:
                ds = cls()
                if _is_p300_like_dataset(ds) or spec.get("task_type") != "p300":
                    USED_MOABB_CLASSES.add(name)
                    print(f"Using MOABB dataset class for {spec['name']}: {name}")
                    return name, ds
            except Exception as e:
                last_error = e
                print(f"Candidate {name} exists but failed to instantiate: {repr(e)}")

    # 2) Public-auto fallback scan. This is intentionally disabled in exact ALS mode.
    if ALLOW_NON_ALS_FALLBACK:
        print(f"Explicit candidates failed for {spec['name']}. Scanning MOABB for a distinct P300-like public dataset...")
        scored = []
        for name in dir(mds):
            if name.startswith("_") or name in USED_MOABB_CLASSES:
                continue
            obj = getattr(mds, name)
            if not inspect.isclass(obj):
                continue
            try:
                ds = obj()
                if not _is_p300_like_dataset(ds):
                    continue
                if not _dataset_matches_keywords(name, ds, include_keywords):
                    # keep as lower priority if no keyword match
                    score = 1
                else:
                    score = 0
                nsub = len(getattr(ds, "subject_list", []) or [])
                scored.append((score, -nsub, name, ds))
            except Exception:
                continue
        if scored:
            scored.sort(key=lambda x: (x[0], x[1], x[2]))
            _, _, name, ds = scored[0]
            USED_MOABB_CLASSES.add(name)
            print(f"Using scanned public P300 dataset for {spec['name']}: {name}; event_id={getattr(ds, 'event_id', {})}")
            return name, ds

    available = [n for n in dir(mds) if not n.startswith("_")]
    raise RuntimeError(
        f"Could not instantiate MOABB dataset for {spec['name']}. "
        f"Tried: {candidates}. Last error: {last_error}. "
        f"Available moabb.datasets symbols include: {available[:120]}"
    )

def make_p300_paradigm(cfg):
    from moabb.paradigms import P300
    attempts = [
        dict(fmin=cfg.FMIN, fmax=cfg.FMAX, tmin=cfg.TMIN, tmax=cfg.TMAX, resample=cfg.RESAMPLE),
        dict(fmin=cfg.FMIN, fmax=cfg.FMAX, tmin=cfg.TMIN, tmax=cfg.TMAX),
        dict(resample=cfg.RESAMPLE),
        dict(),
    ]
    last_error = None
    for kwargs in attempts:
        try:
            print("Trying P300 paradigm kwargs:", kwargs)
            return P300(**kwargs)
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not create MOABB P300 paradigm. Last error: {repr(last_error)}")

def load_moabb_p300_dataset(spec, cfg):
    import moabb
    moabb.set_log_level("WARNING")
    candidates = spec.get("moabb_candidates", cfg.DATASET_CANDIDATES)
    name, dataset = instantiate_moabb_dataset(spec)
    paradigm = make_p300_paradigm(cfg)
    subjects = list(getattr(dataset, "subject_list", []))
    if len(subjects) == 0:
        raise RuntimeError("Dataset has empty subject_list.")
    if cfg.MAX_SUBJECTS is not None:
        subjects = subjects[:cfg.MAX_SUBJECTS]
    print(f"Loading {spec['name']} via MOABB class {name}; subjects: {subjects}")
    X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=subjects)
    y = label_to_binary(labels)
    subjects_arr = metadata["subject"].values if "subject" in metadata.columns else np.repeat(subjects, len(y)//len(subjects))
    X = ensure_trial_tensor(X)
    duration = max(cfg.TMAX - cfg.TMIN, 1e-6)
    sfreq = getattr(paradigm, "resample", None) or cfg.RESAMPLE or int(round(X.shape[-1] / duration))
    return {
        "name": spec["name"],
        "source_name": name,
        "task_type": spec.get("task_type", "p300"),
        "X": X.astype(np.float32),
        "y": np.asarray(y, dtype=np.int64),
        "subjects": np.asarray(subjects_arr),
        "sfreq": int(sfreq),
        "meta": {"moabb_class": name, "description": spec.get("description", "")},
    }


### 4.2 Generic download helpers


In [8]:
def _download_direct_url(url, out_dir, preferred_name="downloaded_dataset.zip"):
    """Download a direct URL and extract it when it is an archive.

    Figshare/UCL ndownloader URLs often end with `/versions/1`, so the URL basename has no
    useful extension. We therefore save those responses as a ZIP by default and extract them.
    """
    import zipfile, tarfile, shutil, requests
    os.makedirs(out_dir, exist_ok=True)
    basename = os.path.basename(url.split("?")[0]).strip()
    if not basename or "." not in basename or basename.isdigit():
        basename = preferred_name
    local = os.path.join(out_dir, basename)
    if not os.path.exists(local):
        print("Downloading:", url)
        with requests.get(url, stream=True, allow_redirects=True, timeout=120) as r:
            r.raise_for_status()
            with open(local, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
    else:
        print("Using existing download:", local)

    # Try archive extraction. If not an archive, leave the file in place.
    try:
        if zipfile.is_zipfile(local):
            with zipfile.ZipFile(local, 'r') as zf:
                zf.extractall(out_dir)
        elif local.endswith(('.tar.gz', '.tgz', '.tar')):
            with tarfile.open(local, 'r:*') as tf:
                tf.extractall(out_dir)
    except Exception as e:
        print("Archive extraction warning:", repr(e))
    return out_dir

def _download_kaggle(slug, out_dir):
    """Download a Kaggle dataset.

    First tries kagglehub, which is convenient in Colab for public datasets. If that fails,
    it falls back to the Kaggle CLI, which requires ~/.kaggle/kaggle.json or env vars.
    """
    import subprocess, os, zipfile, glob, shutil
    os.makedirs(out_dir, exist_ok=True)

    # 1) KaggleHub path-first download.
    try:
        import kagglehub
        print("Trying KaggleHub download:", slug)
        kh_path = kagglehub.dataset_download(slug)
        print("KaggleHub dataset path:", kh_path)
        return kh_path
    except Exception as e:
        print("KaggleHub failed; falling back to Kaggle CLI:", repr(e))

    # 2) Kaggle CLI fallback.
    kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
    if not os.path.exists(kaggle_json):
        if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
            os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
            with open(kaggle_json, "w") as f:
                json.dump({"username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}, f)
            os.chmod(kaggle_json, 0o600)
        else:
            raise RuntimeError(
                "Kaggle credentials not found. In Colab, upload kaggle.json to ~/.kaggle/ "
                "or set KAGGLE_USERNAME and KAGGLE_KEY. KaggleHub also failed."
            )
    cmd = ["kaggle", "datasets", "download", "-d", slug, "-p", out_dir, "--unzip"]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    return out_dir


### 4.3 Generic local file readers


In [9]:
def _load_npz(path):
    data = np.load(path, allow_pickle=True)
    keys = set(data.files)
    def first(*names):
        for n in names:
            if n in keys:
                return data[n]
        return None
    X = first("X", "x", "data", "trials", "epochs")
    y = first("y", "labels", "label", "target", "targets")
    subjects = first("subjects", "subject", "subj", "patient", "patients", "s")
    sfreq = first("sfreq", "fs", "sampling_rate")
    if X is None or y is None:
        raise ValueError(f"NPZ {path} must contain X and y-like arrays. Keys={keys}")
    if subjects is None:
        raise ValueError(f"NPZ {path} must contain subject/patient IDs for LOSO. Keys={keys}")
    sfreq = int(np.asarray(sfreq).item()) if sfreq is not None else int(getattr(cfg, "RESAMPLE", 128))
    y, _ = normalize_label_ids(np.asarray(y).ravel())
    return ensure_trial_tensor(X), y, np.asarray(subjects).ravel(), sfreq

def _load_mat(path):
    import scipy.io
    try:
        mat = scipy.io.loadmat(path)
    except NotImplementedError:
        import mat73
        mat = mat73.loadmat(path)
    # flatten top-level keys only; robust enough for common exported datasets.
    candidates_X = ["X", "x", "data", "trials", "epochs", "EEG", "signals"]
    candidates_y = ["y", "labels", "label", "target", "targets", "class", "classes"]
    candidates_s = ["subjects", "subject", "subj", "patient", "patients", "s"]
    def find(cands):
        for k in cands:
            if k in mat:
                return mat[k]
        return None
    X = find(candidates_X); y = find(candidates_y); subjects = find(candidates_s)
    if X is None or y is None or subjects is None:
        raise ValueError(f"Could not auto-detect X/y/subjects in MAT {path}. Keys={list(mat.keys())[:50]}")
    X = ensure_trial_tensor(np.asarray(X))
    # Heuristic transpose if saved as C x T x N
    if X.shape[0] < X.shape[-1] and X.shape[0] < 256 and X.shape[-1] > 256:
        pass
    y, _ = normalize_label_ids(np.asarray(y).ravel())
    subjects = np.asarray(subjects).ravel()
    sfreq = int(np.asarray(mat.get("sfreq", mat.get("fs", cfg.RESAMPLE))).item()) if ("sfreq" in mat or "fs" in mat) else cfg.RESAMPLE
    return X, y, subjects, sfreq

def _load_csv(path):
    df = pd.read_csv(path)
    label_cols = [c for c in df.columns if c.lower() in ["y", "label", "labels", "target", "class"]]
    subj_cols = [c for c in df.columns if c.lower() in ["subject", "subjects", "subj", "patient", "patients", "s"]]
    if not label_cols or not subj_cols:
        raise ValueError(f"CSV {path} needs subject and label columns. Columns={df.columns.tolist()[:40]}")
    y_raw = df[label_cols[0]].values
    y, _ = normalize_label_ids(y_raw)
    subjects = df[subj_cols[0]].values
    feature_cols = [c for c in df.columns if c not in label_cols + subj_cols]
    X = df[feature_cols].values.astype(np.float32)
    return ensure_trial_tensor(X), y, subjects, cfg.RESAMPLE

def load_generic_folder(folder, spec, cfg):
    import glob
    if not folder or not os.path.exists(folder):
        raise FileNotFoundError(f"Folder/path does not exist: {folder}")
    if os.path.isfile(folder):
        files = [folder]
        base_folder = os.path.dirname(folder)
    else:
        files = []
        for ext in ["*.npz", "*.mat", "*.csv"]:
            files.extend(glob.glob(os.path.join(folder, "**", ext), recursive=True))
        base_folder = folder
    if not files:
        raise FileNotFoundError(
            f"No .npz/.mat/.csv files found in {folder}. "
            "For EDF/GDF/SET datasets, first convert to a standardized NPZ with X,y,subjects,sfreq."
        )
    # Prefer standardized NPZ if present.
    files = sorted(files, key=lambda p: (0 if p.endswith('.npz') else 1 if p.endswith('.mat') else 2, len(p)))
    last_err = None
    for path in files:
        try:
            print(f"Trying to load {spec['name']} from {path}")
            if path.endswith('.npz'):
                X, y, subjects, sfreq = _load_npz(path)
            elif path.endswith('.mat'):
                X, y, subjects, sfreq = _load_mat(path)
            elif path.endswith('.csv'):
                X, y, subjects, sfreq = _load_csv(path)
            else:
                continue
            if len(np.unique(subjects)) < 2:
                raise ValueError("Need at least two subjects/patients for LOSO.")
            return {"name": spec["name"], "source_name": path, "task_type": spec.get("task_type", "mi"),
                    "X": X.astype(np.float32), "y": y.astype(np.int64), "subjects": np.asarray(subjects),
                    "sfreq": int(sfreq), "meta": {"file": path, "description": spec.get("description", "")}}
        except Exception as e:
            last_err = e
            print("  failed:", repr(e))
    raise RuntimeError(f"Could not load {spec['name']} from {folder}. Last error: {last_err}")


### 4.4 UCL MATLAB parsing helpers


In [10]:
def _get_struct_field(obj, name):
    """Get a MATLAB struct field from dict, scipy mat_struct, numpy void, or object."""
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj.get(name, None)
    if hasattr(obj, name):
        return getattr(obj, name)
    if isinstance(obj, np.void) and obj.dtype.names and name in obj.dtype.names:
        return obj[name]
    if isinstance(obj, np.ndarray) and obj.dtype.names and name in obj.dtype.names:
        return obj[name]
    return None

def _has_lrre_fields(obj):
    return (_get_struct_field(obj, "L") is not None) and (_get_struct_field(obj, "R") is not None) and (_get_struct_field(obj, "Re") is not None)

def _find_lrre_subject_structs(obj, max_depth=8):
    """Recursively find UCL subject structs containing fields L, R, and Re."""
    found, seen = [], set()
    def rec(x, depth):
        if depth > max_depth or x is None:
            return
        xid = id(x)
        if xid in seen:
            return
        seen.add(xid)
        if _has_lrre_fields(x):
            found.append(x)
            return
        if isinstance(x, dict):
            for k, v in x.items():
                if not str(k).startswith("__"):
                    rec(v, depth+1)
            return
        if isinstance(x, (list, tuple)):
            for v in x:
                rec(v, depth+1)
            return
        if isinstance(x, np.ndarray):
            if x.dtype == object or x.dtype.names is not None:
                for v in x.ravel():
                    rec(v, depth+1)
            return
    rec(obj, 0)
    return found

def _trial_matrix_to_channels_time(mat):
    arr = np.asarray(mat, dtype=np.float32).squeeze()
    if arr.ndim != 2:
        return None
    # UCL page says each trial matrix has rows=timestamps and columns=channels.
    if arr.shape[0] >= arr.shape[1]:
        arr = arr.T  # [channels, time]
    # Remove obviously empty rows/columns if any.
    if arr.shape[0] < 1 or arr.shape[1] < 2:
        return None
    return arr.astype(np.float32)


In [11]:
def _iter_trials_from_ucl_field(field):
    """Yield [channels, time] trial matrices from a MATLAB cell/object/numeric trial field."""
    if field is None:
        return
    arr = np.asarray(field, dtype=object if isinstance(field, (list, tuple)) else None)
    # Cell/object array: usually one cell per trial.
    if isinstance(field, (list, tuple)) or (isinstance(arr, np.ndarray) and arr.dtype == object):
        for item in np.asarray(field, dtype=object).ravel():
            # Sometimes cells contain nested arrays/lists.
            if isinstance(item, (list, tuple)) or (isinstance(item, np.ndarray) and item.dtype == object):
                for sub in np.asarray(item, dtype=object).ravel():
                    out = _trial_matrix_to_channels_time(sub)
                    if out is not None:
                        yield out
            else:
                out = _trial_matrix_to_channels_time(item)
                if out is not None:
                    yield out
        return

    arr = np.asarray(field)
    if arr.ndim == 2:
        out = _trial_matrix_to_channels_time(arr)
        if out is not None:
            yield out
    elif arr.ndim == 3:
        # Guess dimensions: one channel dimension is usually 8-64, time is often largest,
        # remaining dimension is trials.
        dims = arr.shape
        possible_c = [i for i, d in enumerate(dims) if 4 <= d <= 128]
        if possible_c:
            ch_dim = min(possible_c, key=lambda i: dims[i])
        else:
            ch_dim = 1
        time_dim = int(np.argmax(dims))
        trial_dim = [i for i in range(3) if i not in [ch_dim, time_dim]][0]
        arr_tct = np.moveaxis(arr, [trial_dim, ch_dim, time_dim], [0, 1, 2])
        for k in range(arr_tct.shape[0]):
            out = np.asarray(arr_tct[k], dtype=np.float32)
            if out.ndim == 2 and out.shape[0] >= 1 and out.shape[1] >= 2:
                yield out

def _stack_trials_with_common_shape(trials):
    if not trials:
        raise ValueError("No valid trials found after parsing.")
    min_c = min(t.shape[0] for t in trials)
    min_t = min(t.shape[1] for t in trials)
    cropped = [t[:min_c, :min_t] for t in trials]
    return np.stack(cropped, axis=0).astype(np.float32)

def _ucl_patient_id_from_path(path):
    """Infer a stable patient/session ID from UCL MAT file path.

    The previous parser treated each small MATLAB L/R struct as a separate subject, creating
    invalid folds with only one left and one right trial. For the UCL ALS dataset, the safe
    default is patient/file-level grouping: all L/R trials parsed from one patient file share
    the same subject ID.
    """
    parts = re.split(r"[\\/]", str(path))
    # Prefer explicit ALS/patient/subject identifiers in file or parent names.
    for part in reversed(parts):
        stem = os.path.splitext(part)[0]
        m = re.search(r"(ALS\s*\d+|patient\s*\d+|subject\s*\d+|subj\s*\d+|S\d+|P\d+)", stem, flags=re.IGNORECASE)
        if m:
            return re.sub(r"\s+", "", m.group(1)).upper()
    # Fallback: file stem.
    stem = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("_") or f"ucl_file_{abs(hash(path))%100000}"

def _collect_ucl_trials_from_structs(subj_structs, spec):
    """Collect L/R(/Re) trials from all LRRe structs inside a patient file."""
    trials_all, labels_all = [], []
    for st in subj_structs:
        L = list(_iter_trials_from_ucl_field(_get_struct_field(st, "L")))
        R = list(_iter_trials_from_ucl_field(_get_struct_field(st, "R")))
        Re = list(_iter_trials_from_ucl_field(_get_struct_field(st, "Re")))
        if spec.get("mi_task", "left_right") == "left_right":
            trials = L + R
            labels = [0] * len(L) + [1] * len(R)
        else:
            trials = L + R + Re
            labels = [0] * len(L) + [1] * len(R) + [2] * len(Re)
        trials_all.extend(trials)
        labels_all.extend(labels)
    return trials_all, labels_all


### 4.5 Archive and file discovery


In [12]:
def _extract_archives_recursively(folder, max_passes=3):
    """Extract zip/tar archives recursively because Figshare/Kaggle archives can be nested."""
    import glob, zipfile, tarfile
    for _ in range(max_passes):
        archives = []
        archives.extend(glob.glob(os.path.join(folder, "**", "*.zip"), recursive=True))
        archives.extend(glob.glob(os.path.join(folder, "**", "*.tar"), recursive=True))
        archives.extend(glob.glob(os.path.join(folder, "**", "*.tar.gz"), recursive=True))
        archives.extend(glob.glob(os.path.join(folder, "**", "*.tgz"), recursive=True))
        changed = False
        for arch in archives:
            marker = arch + ".extracted"
            if os.path.exists(marker):
                continue
            try:
                target = os.path.splitext(arch)[0] + "_extracted"
                os.makedirs(target, exist_ok=True)
                if zipfile.is_zipfile(arch):
                    with zipfile.ZipFile(arch, "r") as zf:
                        zf.extractall(target)
                    open(marker, "w").write("ok")
                    changed = True
                elif arch.lower().endswith((".tar", ".tar.gz", ".tgz")):
                    with tarfile.open(arch, "r:*") as tf:
                        tf.extractall(target)
                    open(marker, "w").write("ok")
                    changed = True
            except Exception as e:
                print("Archive extraction warning for", arch, ":", repr(e))
        if not changed:
            break
    return folder

def _case_insensitive_files(folder, extensions):
    import glob
    files = []
    for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
        if os.path.isfile(p) and p.lower().endswith(tuple(e.lower() for e in extensions)):
            files.append(p)
    return sorted(files)


### 4.6 Robust UCL public-file downloader


In [13]:
class DatasetUnavailableError(RuntimeError):
    """Expected external dataset is temporarily inaccessible or not configured."""

def _is_valid_binary_download(path, min_bytes=1024):
    """Reject empty files and HTML error pages saved with a data extension."""
    if not os.path.isfile(path) or os.path.getsize(path) < min_bytes:
        return False
    with open(path, "rb") as handle:
        head = handle.read(512).lower()
    return not any(token in head for token in (b"<html", b"<!doctype html", b"access denied", b"forbidden"))

def _figshare_urls_from_datacite(doi="10.5522/04/28156016.v1"):
    """Resolve file URLs from DOI metadata; failure is non-fatal."""
    import requests

    api = f"https://api.datacite.org/dois/{doi}"
    response = requests.get(api, timeout=60, headers={"Accept": "application/vnd.api+json"})
    response.raise_for_status()
    attrs = response.json().get("data", {}).get("attributes", {})
    urls = []
    for item in attrs.get("relatedIdentifiers", []) or []:
        value = str(item.get("relatedIdentifier", ""))
        if item.get("relationType") == "HasPart" and "/files/" in value:
            urls.append(value.replace("http://", "https://"))
    return list(dict.fromkeys(urls))

def _figshare_urls_from_public_api(article_id, version=1):
    """Use the public Figshare API when the institutional mirror permits it."""
    import requests

    endpoints = (
        f"https://api.figshare.com/v2/articles/{article_id}/versions/{version}/files",
        f"https://api.figshare.com/v2/articles/{article_id}/files",
        f"https://api.figshare.com/v2/articles/{article_id}/versions/{version}",
        f"https://api.figshare.com/v2/articles/{article_id}",
    )
    errors = []
    for endpoint in endpoints:
        try:
            response = requests.get(endpoint, timeout=60, headers={"Accept": "application/json"})
            response.raise_for_status()
            payload = response.json()
            files = payload if isinstance(payload, list) else payload.get("files", [])
            urls = [f.get("download_url") for f in files if isinstance(f, dict) and f.get("download_url")]
            if urls:
                return list(dict.fromkeys(urls))
        except Exception as exc:
            errors.append(f"{endpoint}: {exc!r}")
    raise DatasetUnavailableError("Figshare public API did not expose files. " + " | ".join(errors[-2:]))

def _candidate_ucl_file_urls(spec):
    """Return stable per-file URLs; article-level ndownloader URLs are intentionally avoided."""
    urls = []
    for file_id in spec.get("figshare_file_ids", ()):
        urls.append(f"https://ndownloader.figshare.com/files/{int(file_id)}")

    try:
        urls.extend(_figshare_urls_from_datacite())
    except Exception as exc:
        print("DataCite file discovery unavailable:", repr(exc))

    try:
        urls.extend(_figshare_urls_from_public_api(spec.get("figshare_article_id", "28156016"), version=1))
    except Exception as exc:
        print("Figshare API file discovery unavailable:", repr(exc))

    # Canonicalize by Figshare file ID so API/DataCite aliases cannot duplicate patients.
    canonical = {}
    for url in urls:
        match = re.search(r"/files/(\d+)", str(url))
        key = match.group(1) if match else str(url)
        canonical.setdefault(key, str(url))
    return list(canonical.values())

def _download_ucl_mat_files(spec, out_dir):
    """Download the eight public patient files independently with retries and validation."""
    import requests

    os.makedirs(out_dir, exist_ok=True)
    urls = _candidate_ucl_file_urls(spec)
    if not urls:
        raise DatasetUnavailableError("No UCL per-file download URLs could be resolved.")

    successes, failures = [], []
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; MVQAN-research/1.0)", "Accept": "*/*"})

    for index, url in enumerate(urls, start=1):
        match = re.search(r"/files/(\d+)", url)
        file_id = match.group(1) if match else f"resolved_{index:02d}"
        local = os.path.join(out_dir, f"ucl_patient_{index:02d}_{file_id}.mat")
        if _is_valid_binary_download(local):
            successes.append(local)
            continue

        last_error = None
        for attempt in range(1, 4):
            try:
                print(f"Downloading UCL file {index}/{len(urls)} (attempt {attempt}): {url}")
                with session.get(url, stream=True, timeout=(30, 300), allow_redirects=True) as response:
                    response.raise_for_status()
                    content_type = str(response.headers.get("content-type", "")).lower()
                    if "text/html" in content_type:
                        raise DatasetUnavailableError(f"Server returned HTML instead of a data file: {url}")
                    tmp = local + ".part"
                    with open(tmp, "wb") as handle:
                        for chunk in response.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                handle.write(chunk)
                    os.replace(tmp, local)
                if not _is_valid_binary_download(local):
                    raise DatasetUnavailableError(f"Downloaded file failed validation: {local}")
                successes.append(local)
                break
            except Exception as exc:
                last_error = exc
                for candidate in (local, local + ".part"):
                    if os.path.exists(candidate):
                        os.remove(candidate)
                time.sleep(min(2 ** (attempt - 1), 4))
        else:
            failures.append((url, repr(last_error)))

    # The official release has eight patient files. Do not silently continue with a partial cohort.
    unique_files = sorted(set(successes))
    if len(unique_files) < 8:
        detail = "; ".join(f"{u} -> {e}" for u, e in failures[:3])
        raise DatasetUnavailableError(
            f"Only {len(unique_files)}/8 UCL patient files were available. {detail}. "
            "Upload the eight .mat files to /content/ucl_als_eeg or set UCL_ALS_LOCAL_PATH."
        )
    return out_dir


In [14]:
def _prepare_ucl_dataset_folder(spec):
    """Prefer manual/cache data, then download stable per-file URLs; never use the failing article ZIP route."""
    manual = str(spec.get("local_path", "") or "").strip()
    candidates = [manual, "/content/ucl_als_eeg"] if manual else ["/content/ucl_als_eeg"]
    for folder in candidates:
        if folder and os.path.isdir(folder):
            mats = _case_insensitive_files(folder, [".mat"])
            if len(mats) >= 8:
                print(f"Using local UCL dataset: {folder} ({len(mats)} MAT files)")
                return folder

    cache = os.path.join("/content", "advisor_datasets", spec["name"].replace(" ", "_"))
    os.makedirs(cache, exist_ok=True)
    cached = _case_insensitive_files(cache, [".mat"])
    if len(cached) >= 8:
        print(f"Using cached UCL dataset: {cache} ({len(cached)} MAT files)")
        return cache

    return _download_ucl_mat_files(spec, cache)


### 4.7 UCL ALS motor-imagery loader


In [15]:
def load_ucl_figshare_als_mi_dataset(spec, cfg):
    """Final UCL ALS EEG loader with patient/file-level subject grouping.

    This fixes the invalid fold issue where the test subject had only {0:1,1:1}. The loader
    now groups all L/R trials parsed from each patient MAT file under one subject ID. If a
    file contains only tiny per-trial structs, they are aggregated at file level.
    """
    article_id = spec.get("figshare_article_id", "28156016")
    folder = _prepare_ucl_dataset_folder(spec)

    import scipy.io
    mat_files = _case_insensitive_files(folder, [".mat"])
    print(f"UCL MAT files discovered: {len(mat_files)}")
    if len(mat_files) == 0:
        sample = _case_insensitive_files(folder, [".zip", ".csv", ".txt", ".xlsx", ".mat"])
        raise FileNotFoundError(f"No .mat files found after UCL download in {folder}. Sample files={sample[:20]}")

    patient_trials = {}
    patient_labels = {}
    last_error = None
    for mat_path in sorted(mat_files):
        print("Trying UCL MAT file:", mat_path)
        try:
            try:
                mat = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
            except NotImplementedError:
                import mat73
                mat = mat73.loadmat(mat_path)
            subj_structs = _find_lrre_subject_structs(mat)
            if not subj_structs:
                print("  no L/R/Re subject structs found in this file")
                continue
            trials, labels = _collect_ucl_trials_from_structs(subj_structs, spec)
            if not trials:
                print("  L/R structs found but no valid trial matrices extracted")
                continue
            pid = _ucl_patient_id_from_path(mat_path)
            patient_trials.setdefault(pid, []).extend(trials)
            patient_labels.setdefault(pid, []).extend(labels)
            cls_counts = dict(zip(*np.unique(np.asarray(labels, dtype=int), return_counts=True)))
            print(f"  grouped {len(trials)} trials under patient/file subject {pid}; class counts={cls_counts}; LRRe structs={len(subj_structs)}")
        except Exception as e:
            last_error = e
            print("  failed:", repr(e))

    if not patient_trials:
        raise RuntimeError(f"Could not parse UCL ALS EEG dataset. Last error: {last_error}")

    # Remove patient files that do not contain both L and R classes for the binary MI task.
    kept_trials, kept_labels, kept_subjects = [], [], []
    dropped = []
    for pid in sorted(patient_trials.keys(), key=str):
        labs = np.asarray(patient_labels[pid], dtype=np.int64)
        if len(np.unique(labs)) < 2:
            dropped.append(pid)
            continue
        kept_trials.extend(patient_trials[pid])
        kept_labels.extend(labs.tolist())
        kept_subjects.extend([pid] * len(labs))
    if dropped:
        print("Dropped UCL patient files without both binary classes:", dropped)
    if not kept_trials:
        raise RuntimeError("UCL loader found no patient/file with both left and right MI trials.")

    X = _stack_trials_with_common_shape(kept_trials)
    y = np.asarray(kept_labels, dtype=np.int64)
    subjects = np.asarray(kept_subjects, dtype=object)
    if len(np.unique(subjects)) < 2:
        raise ValueError(
            "UCL loader still found fewer than two patient/file subjects. This means the public archive layout "
            "does not expose patient-level MAT files clearly; inspect the printed file list before reporting results."
        )
    print("UCL patient/file trial counts:", pd.Series(subjects).value_counts().to_dict())
    print("UCL class counts:", dict(zip(*np.unique(y, return_counts=True))))
    sfreq = int(spec.get("sfreq", getattr(cfg, "RESAMPLE", 128)))
    return {
        "name": spec["name"],
        "source_name": f"Figshare article {article_id}",
        "task_type": spec.get("task_type", "mi"),
        "X": X.astype(np.float32),
        "y": y.astype(np.int64),
        "subjects": subjects,
        "sfreq": sfreq,
        "meta": {
            "description": spec.get("description", ""),
            "mi_task": spec.get("mi_task", "left_right"),
            "subject_grouping": "patient/file-level grouping",
            "n_mat_files": len(mat_files),
        },
    }


### 4.8 EEGET parsing helpers


In [16]:
def _infer_eeget_subject_label(path):
    parts = re.split(r"[\\/]", path)
    subject = None
    label = None
    for part in parts:
        if re.fullmatch(r"ALS\d+", part, flags=re.IGNORECASE):
            subject = part.upper()
            label = 1
            break
        if re.fullmatch(r"id\d+", part, flags=re.IGNORECASE):
            subject = part.lower()
            label = 0
            break
    return subject, label

def _estimate_sfreq_from_timestamp(ts, fallback=128):
    try:
        arr = pd.to_numeric(pd.Series(ts), errors="coerce").dropna().values.astype(float)
        if len(arr) < 3:
            return int(fallback)
        diffs = np.diff(arr)
        diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
        if len(diffs) == 0:
            return int(fallback)
        med = float(np.median(diffs))
        # If timestamps are milliseconds, median difference is often > 1.
        sf = 1000.0 / med if med > 1.0 else 1.0 / med
        if not np.isfinite(sf) or sf < 4 or sf > 2048:
            return int(fallback)
        return int(round(sf))
    except Exception:
        return int(fallback)

def _read_numeric_timeseries_csv(path, allow_et=False, fallback_sfreq=128):
    df = pd.read_csv(path)
    cols_lower = {c: c.lower().strip() for c in df.columns}
    timestamp_col = None
    for c, lc in cols_lower.items():
        if lc in ["timestamp", "time", "time_stamp", "timestamps"]:
            timestamp_col = c
            break
    sfreq = _estimate_sfreq_from_timestamp(df[timestamp_col].values, fallback_sfreq) if timestamp_col else int(fallback_sfreq)

    numeric_cols = []
    for c in df.columns:
        lc = cols_lower[c]
        if c == timestamp_col:
            continue
        # Drop textual typing/sentence columns.
        if any(t in lc for t in ["sentence", "character", "typing", "label", "class"]):
            continue
        ser = pd.to_numeric(df[c], errors="coerce")
        if ser.notna().mean() > 0.80:
            numeric_cols.append(c)
    if not numeric_cols:
        raise ValueError(f"No usable numeric signal columns in {path}. Columns={df.columns.tolist()[:30]}")

    base = os.path.basename(path).lower()
    is_et = base == "et.csv" or base.startswith("et")
    if is_et and not allow_et:
        raise ValueError("ET.csv detected but ET fallback is disabled.")
    if (not is_et) and len(numeric_cols) < 2:
        raise ValueError(f"Too few numeric signal columns for EEG-like CSV {path}: {numeric_cols}")

    arr = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    arr = arr.interpolate(limit_direction="both").fillna(0.0).values.astype(np.float32)
    # Return [channels, time]
    return arr.T, sfreq, numeric_cols, ("ET-fallback" if is_et else "EEG-like-CSV")

def _segment_channels_time(Xct, sfreq, window_seconds=2.0, stride_seconds=2.0):
    C, T = Xct.shape
    win = max(8, int(round(float(window_seconds) * sfreq)))
    stride = max(1, int(round(float(stride_seconds) * sfreq)))
    if T <= win:
        return [Xct[:, :T]]
    out = []
    for start in range(0, T - win + 1, stride):
        out.append(Xct[:, start:start+win])
    return out if out else [Xct[:, :win]]


### 4.9 EEGET dataset loader


In [17]:
def load_eeget_als_dataset(spec, cfg):
    """Final EEGET loader: path-inferred labels, fixed per-subject cap, explicit modality reporting.

    The remaining class imbalance is handled at fold-training time by balanced source-index sampling.
    """
    base = os.path.join("/content", "advisor_datasets", spec["name"].replace(" ", "_"))
    if spec.get("local_path"):
        folder = spec["local_path"]
    elif spec.get("kaggle_slug"):
        folder = _download_kaggle(spec["kaggle_slug"], base)
    elif spec.get("direct_url"):
        folder = _download_direct_url(spec["direct_url"], base, preferred_name=f"{spec['name']}.zip")
    else:
        raise RuntimeError("EEGET-ALS source not configured.")
    _extract_archives_recursively(folder)

    csv_files = _case_insensitive_files(folder, [".csv"])
    print(f"EEGET CSV files discovered: {len(csv_files)}")
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in EEGET folder {folder}")

    eeg_like, et_files = [], []
    for p in csv_files:
        subj, lab = _infer_eeget_subject_label(p)
        if subj is None:
            continue
        b = os.path.basename(p).lower()
        full = p.lower()
        if b == "et.csv" or b.startswith("et"):
            et_files.append(p)
        elif any(tok in full for tok in ["eeg", "brain", "openbci", "signal", "signals", "data"]):
            eeg_like.append(p)
        else:
            # keep generic numeric files as EEG-like candidates, but ET.csv remains separate
            eeg_like.append(p)

    allow_et = bool(spec.get("allow_et_fallback", False))
    candidate_files = eeg_like
    modality_note = "EEG-like-CSV"
    if len(candidate_files) == 0 and allow_et:
        print("[WARNING] No EEG-like CSV files were found. Using ET.csv fallback for EEGET-ALS.")
        print("[WARNING] If this fallback is used, report it as ET/multimodal proxy, not EEG-only.")
        candidate_files = et_files
        modality_note = "ET-fallback"
    if len(candidate_files) == 0:
        raise RuntimeError("No EEG-like EEGET CSV files found. Provide standardized EEG files or enable explicit ET fallback.")

    print(f"EEGET selected files: {len(candidate_files)} | modality={modality_note}")
    max_per_subj = int(spec.get("max_windows_per_subject", 160))
    trials, labels, subjects, sfreqs = [], [], [], []
    counts_by_subject = {}
    last_err = None
    for p in sorted(candidate_files):
        subj, lab = _infer_eeget_subject_label(p)
        if subj is None or lab is None:
            continue
        if counts_by_subject.get(subj, 0) >= max_per_subj:
            continue
        try:
            Xct, sf, used_cols, mod = _read_numeric_timeseries_csv(
                p, allow_et=(modality_note == "ET-fallback"), fallback_sfreq=int(spec.get("sfreq", cfg.RESAMPLE))
            )
            # robust per-file channel normalization before windowing
            Xct = Xct - Xct.mean(axis=1, keepdims=True)
            Xct = Xct / (Xct.std(axis=1, keepdims=True) + 1e-6)
            segs = _segment_channels_time(
                Xct, sf, window_seconds=float(spec.get("window_seconds", 2.0)), stride_seconds=float(spec.get("stride_seconds", 2.0))
            )
            for seg in segs:
                if counts_by_subject.get(subj, 0) >= max_per_subj:
                    break
                trials.append(seg.astype(np.float32)); labels.append(int(lab)); subjects.append(subj); sfreqs.append(sf)
                counts_by_subject[subj] = counts_by_subject.get(subj, 0) + 1
        except Exception as e:
            last_err = e
            if len(trials) == 0:
                print("  skipped", p, "::", repr(e))
            continue
    if not trials:
        raise RuntimeError(f"Could not parse any EEGET-ALS trials. Last error={last_err}")
    X = _stack_trials_with_common_shape(trials)
    y = np.asarray(labels, dtype=np.int64)
    subj_arr = np.asarray(subjects, dtype=object)
    sfreq = int(np.median(sfreqs)) if sfreqs else int(spec.get("sfreq", cfg.RESAMPLE))
    if len(np.unique(subj_arr)) < 2 or len(np.unique(y)) < 2:
        raise ValueError(f"EEGET loader needs at least two subjects and two labels. subjects={np.unique(subj_arr)}, labels={np.unique(y)}")
    print("EEGET subject window counts:", counts_by_subject)
    print("EEGET class counts before fold balancing:", dict(zip(*np.unique(y, return_counts=True))))
    return {
        "name": spec["name"], "source_name": spec.get("kaggle_slug", "EEGET-ALS Kaggle"),
        "task_type": spec.get("task_type", "als_control"),
        "X": X.astype(np.float32), "y": y.astype(np.int64), "subjects": subj_arr, "sfreq": sfreq,
        "meta": {
            "description": spec.get("description", ""),
            "label_mapping": "id*=0/control-like, ALS*=1/ALS",
            "modality_used": modality_note,
            "warning": "ET fallback is not EEG-only" if modality_note == "ET-fallback" else "",
            "fold_training_balance": "enabled downstream for als_control",
        },
    }


### 4.10 Dataset dispatch


In [18]:
def _make_binary_if_needed(X, y_raw, subjects, spec):
    """Standardize labels for the current binary-focused pipeline.

    If labels are already binary, normalize them to 0/1. If more than two classes are found and
    auto_make_binary=True, keep the two largest non-rest-like classes. This is a safe runnable default,
    but label meaning should be verified before final paper writing.
    """
    y_arr = np.asarray(y_raw).ravel()
    y_str = y_arr.astype(str)
    unique = np.unique(y_str)
    if len(unique) == 2:
        vals = sorted(unique.tolist())
        mapping = {vals[0]: 0, vals[1]: 1}
        return X, np.asarray([mapping[v] for v in y_str], dtype=np.int64), subjects, mapping
    # Prefer ALS/control mapping when labels are textual.
    low = np.char.lower(y_str.astype(str))
    als_mask = np.array([("als" in s) or ("patient" in s) for s in low])
    control_mask = np.array([("control" in s) or ("healthy" in s) or ("hc" == s.strip()) for s in low])
    if als_mask.any() and control_mask.any():
        keep = als_mask | control_mask
        y_bin = als_mask[keep].astype(np.int64)
        return X[keep], y_bin, np.asarray(subjects)[keep], {"control/healthy": 0, "ALS/patient": 1}
    if spec.get("auto_make_binary", False):
        # Exclude rest-like labels if detectable, then keep two largest classes.
        rest_like = np.array([s.lower() in ["re", "rest", "idle", "baseline"] for s in y_str])
        candidate_labels = y_str[~rest_like] if (~rest_like).sum() >= 2 else y_str
        vals, counts = np.unique(candidate_labels, return_counts=True)
        order = np.argsort(-counts)
        keep_vals = vals[order[:2]]
        keep = np.isin(y_str, keep_vals)
        mapping = {keep_vals[0]: 0, keep_vals[1]: 1}
        y_bin = np.asarray([mapping[v] for v in y_str[keep]], dtype=np.int64)
        print(f"[auto_make_binary] {spec['name']}: kept two classes {mapping}; dropped {len(y_str)-keep.sum()} samples")
        return X[keep], y_bin, np.asarray(subjects)[keep], mapping
    y, mapping = normalize_label_ids(y_arr)
    if len(mapping) != 2:
        raise ValueError(f"Dataset {spec['name']} has {len(mapping)} classes. Set auto_make_binary=True or implement multiclass metrics.")
    return X, y, subjects, mapping

def load_kaggle_or_url_or_local_dataset(spec, cfg):
    # Dataset-specific EEGET loader fixes nested Kaggle structure.
    if "eeget" in spec.get("name", "").lower() or "eeget" in spec.get("kaggle_slug", "").lower():
        return load_eeget_als_dataset(spec, cfg)

    base = os.path.join("/content", "advisor_datasets", spec["name"].replace(" ", "_"))
    local_path = spec.get("local_path") or ""
    if local_path:
        ds = load_generic_folder(local_path, spec, cfg)
    elif spec.get("kaggle_slug"):
        folder = _download_kaggle(spec["kaggle_slug"], base)
        ds = load_generic_folder(folder, spec, cfg)
    elif spec.get("direct_url"):
        folder = _download_direct_url(spec["direct_url"], base, preferred_name=f"{spec['name']}.zip")
        ds = load_generic_folder(folder, spec, cfg)
    else:
        raise RuntimeError(
            f"No online source configured for {spec['name']}. "
            "Please fill kaggle_slug, direct_url, or local_path in ADVISOR_DATASETS."
        )
    Xb, yb, sb, mapping = _make_binary_if_needed(ds["X"], ds["y"], ds["subjects"], spec)
    ds["X"], ds["y"], ds["subjects"] = Xb.astype(np.float32), yb.astype(np.int64), np.asarray(sb)
    ds.setdefault("meta", {})["label_mapping"] = str(mapping)
    return ds

def load_one_advisor_dataset(spec, cfg):
    if spec["source_type"] == "moabb":
        return load_moabb_p300_dataset(spec, cfg)
    elif spec["source_type"] == "ucl_figshare_als_mi":
        return load_ucl_figshare_als_mi_dataset(spec, cfg)
    elif spec["source_type"] == "kaggle_or_url_or_local":
        return load_kaggle_or_url_or_local_dataset(spec, cfg)
    else:
        raise ValueError(f"Unknown source_type={spec['source_type']} for {spec['name']}")


### 4.11 Availability-aware dataset orchestration


In [19]:
def load_all_advisor_datasets(cfg):
    """Load every active source that is available.

    Quick/tune modes continue with available datasets and write a missing-source report.
    Paper mode is strict and raises when an active required dataset is unavailable.
    """
    loaded, missing = [], []
    active = set(getattr(cfg, "ACTIVE_DATASETS", ()) or ())

    for spec in ADVISOR_DATASETS:
        if active and spec["name"] not in active:
            print(f"[SKIP by ACTIVE_DATASETS] {spec['name']}")
            continue
        print("\n" + "=" * 90)
        print("Loading advisor dataset:", spec["name"])
        try:
            dataset = load_one_advisor_dataset(spec, cfg)
            classes = dict(zip(*np.unique(dataset["y"], return_counts=True)))
            print(
                f"Loaded {dataset['name']} | X={dataset['X'].shape} | classes={classes} | "
                f"subjects={np.unique(dataset['subjects'])} | sfreq={dataset['sfreq']}"
            )
            loaded.append(dataset)
        except Exception as exc:
            message = repr(exc)
            print(f"[UNAVAILABLE] {spec['name']}: {message}")
            missing.append({"dataset": spec["name"], "error": message})

    os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
    report_path = os.path.join(cfg.RESULTS_DIR, "dataset_loading_report.json")
    with open(report_path, "w", encoding="utf-8") as handle:
        json.dump(
            {
                "run_mode": cfg.RUN_MODE,
                "strict": bool(REQUIRE_ALL_ADVISOR_DATASETS),
                "loaded": [item["name"] for item in loaded],
                "missing": missing,
            },
            handle,
            indent=2,
        )
    print("Dataset loading report:", report_path)

    if missing:
        print("\nUnavailable active datasets:")
        for item in missing:
            print(" -", item["dataset"], "::", item["error"])
        if REQUIRE_ALL_ADVISOR_DATASETS:
            raise RuntimeError(
                "Paper mode requires every active dataset. Supply the missing local files, then rerun the loader cells."
            )
        print("Continuing because quick/tune mode is non-strict.")

    if not loaded:
        raise RuntimeError("No active dataset loaded. Configure at least one accessible source.")
    return loaded


In [20]:
advisor_loaded_datasets = load_all_advisor_datasets(cfg)



Loading advisor dataset: BNCI-P300


Using MOABB dataset class for BNCI-P300: BNCI2014_009
Trying P300 paradigm kwargs: {'fmin': 0.1, 'fmax': 30.0, 'tmin': 0.0, 'tmax': 0.8, 'resample': 128}
Loading BNCI-P300 via MOABB class BNCI2014_009; subjects: [1, 2, 3, 4, 5, 6]


  0%|                                              | 0.00/18.5M [00:00<?, ?B/s]

SHA256 hash of downloaded file: beddf78f1834ddef15553e32c9d18c46bc9b3fd244ef3a8e2fe362066dfb027d
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


  0%|                                              | 0.00/18.5M [00:00<?, ?B/s]

...

  0%|                                              | 0.00/18.5M [00:00<?, ?B/s]

SHA256 hash of downloaded file: 46f42354ec901ea24a3a5f5a7755af7e3bedc5b3f1ac5d0094f6874f9132e53c
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


  0%|                                              | 0.00/18.5M [00:00<?, ?B/s]

SHA256 hash of downloaded file: fb8045eae01d52bff6c0d9eff3233992e579fd12d4253ea3bbb5a1e64ea19081
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


Loaded BNCI-P300 | X=(10368, 16, 103) | classes={np.int64(0): np.int64(8640), np.int64(1): np.int64(1728)} | subjects=[1 2 3 4 5 6] | sfreq=128

Loading advisor dataset: UCL-ALS-EEG
UCL MAT files discovered: 8
Trying UCL MAT file: /content/advisor_datasets/UCL-ALS-EEG/ucl_patient_01_51526682.mat
  grouped 320 trials under patient/file subject ucl_patient_01_51526682; class counts={np.int64(0): np.int64(160), np.int64(1): np.int64(160)}; LRRe structs=160
Trying UCL MAT file: /content/advisor_datasets/UCL-ALS-EEG/ucl_patient_02_51526688.mat
  grouped 344 trials under patient/file subject ucl_patient_02_51526688; class counts={np.int64(0): np.int64(172), np.int64(1): np.int64(172)}; LRRe structs=172
Trying UCL MAT file: /content/advisor_datasets/UCL-ALS-EEG/ucl_patient_03_51526691.mat
  grouped 318 trials under patient/file subject ucl_patient_03_51526691; class counts={np.int64(0): np.int64(159), np.int64(1): np.int64(159)}; LRRe structs=159
Trying UCL MAT file: /content/advisor_datasets

# 6. Structured EEG views


In [21]:

def _safe_std(x, axis=None, keepdims=False):
    return np.std(x, axis=axis, keepdims=keepdims) + 1e-6


def _time_to_idx(t0, t1, T, tmin, tmax):
    duration = max(tmax - tmin, 1e-8)
    i0 = int(np.floor((t0 - tmin) / duration * T))
    i1 = int(np.ceil((t1 - tmin) / duration * T))
    i0 = max(0, min(T - 1, i0)); i1 = max(i0 + 1, min(T, i1))
    return np.arange(i0, i1)


def temporal_erp_view(X, cfg):
    N, C, T = X.shape
    windows = [(cfg.TMIN, min(0.20, cfg.TMAX)), (max(cfg.TMIN,0.20), min(0.40,cfg.TMAX)),
               (max(cfg.TMIN,0.40), min(0.60,cfg.TMAX)), (max(cfg.TMIN,0.60), cfg.TMAX)]
    feats = []
    for t0,t1 in windows:
        idx = _time_to_idx(t0,t1,T,cfg.TMIN,cfg.TMAX); seg = X[:,:,idx]
        feats += [seg.mean(-1), seg.std(-1), seg.max(-1), seg.min(-1)]
    return np.concatenate(feats, axis=1).astype(np.float32)


def temporal_general_view(X, n_windows=4):
    N,C,T=X.shape; feats=[]
    for idx in np.array_split(np.arange(T), n_windows):
        seg=X[:,:,idx]
        feats += [seg.mean(-1), seg.std(-1), seg.max(-1), seg.min(-1)]
    return np.concatenate(feats, axis=1).astype(np.float32)


def spatial_region_view(X, n_regions=4, n_windows=4):
    N,C,T=X.shape; feats=[]
    for ch_idx in np.array_split(np.arange(C), min(n_regions,C)):
        for t_idx in np.array_split(np.arange(T), n_windows):
            seg=X[:,ch_idx][:,:,t_idx]
            feats += [seg.mean((1,2))[:,None], seg.std((1,2))[:,None], seg.max((1,2))[:,None]]
    return np.concatenate(feats, axis=1).astype(np.float32)


def frequency_band_view(X, sfreq, bands=((0.5,4),(4,8),(8,13),(13,30))):
    N,C,T=X.shape
    freqs=np.fft.rfftfreq(T, d=1.0/max(float(sfreq), 1.0)); fft=np.fft.rfft(X, axis=-1)
    psd=(np.abs(fft)**2)/max(T,1); feats=[]
    for lo,hi in bands:
        mask=(freqs>=lo)&(freqs<=hi)
        bp=psd[:,:,mask].mean(-1) if mask.sum()>0 else np.zeros((N,C), dtype=np.float32)
        feats.append(np.log1p(bp))
    return np.concatenate(feats, axis=1).astype(np.float32)


def erp_stat_view(X, cfg):
    N,C,T=X.shape
    early=_time_to_idx(cfg.TMIN,min(0.20,cfg.TMAX),T,cfg.TMIN,cfg.TMAX)
    p300a=_time_to_idx(max(cfg.TMIN,0.20),min(0.40,cfg.TMAX),T,cfg.TMIN,cfg.TMAX)
    p300b=_time_to_idx(max(cfg.TMIN,0.40),min(0.60,cfg.TMAX),T,cfg.TMIN,cfg.TMAX)
    late=_time_to_idx(max(cfg.TMIN,0.60),cfg.TMAX,T,cfg.TMIN,cfg.TMAX)
    feats=[]
    for idx in [early,p300a,p300b,late]:
        seg=X[:,:,idx]; feats += [seg.mean((1,2))[:,None], seg.std((1,2))[:,None], np.abs(seg).max((1,2))[:,None]]
    feats.append((X[:,:,np.concatenate([p300a,p300b,late])].mean((1,2)) - X[:,:,early].mean((1,2)))[:,None])
    return np.concatenate(feats, axis=1).astype(np.float32)


def logvar_stat_view(X):
    # MI/ALS robust amplitude/log-variance summary; avoids ERP-specific time assumptions.
    eps=1e-6
    lv=np.log(np.var(X, axis=-1)+eps)
    mean= X.mean(axis=-1); std=X.std(axis=-1)
    global_feats=np.stack([X.mean((1,2)), X.std((1,2)), np.abs(X).max((1,2))], axis=1)
    return np.concatenate([lv, mean, std, global_feats], axis=1).astype(np.float32)


def covariance_view(X, eps=1e-3):
    N,C,T=X.shape; iu=np.triu_indices(C); feats=np.zeros((N,len(iu[0])), dtype=np.float32)
    for i in range(N):
        Xi=X[i]-X[i].mean(axis=1, keepdims=True)
        cov=(Xi@Xi.T)/max(T-1,1)
        scale=np.trace(cov)/max(C,1)
        cov += eps * (scale if np.isfinite(scale) and scale>0 else 1.0) * np.eye(C)
        cov = cov / (np.trace(cov)+1e-8)
        feats[i]=cov[iu]
    return feats.astype(np.float32)


def build_views(X, sfreq, cfg, task_type="p300"):
    task_type=str(task_type).lower()
    if task_type == "p300":
        return {
            "temporal_erp": temporal_erp_view(X, cfg),
            "spatial_region": spatial_region_view(X, n_regions=4, n_windows=4),
            "frequency_band": frequency_band_view(X, sfreq, bands=((0.5,4),(4,8),(8,13),(13,30))),
            "erp_stat": erp_stat_view(X, cfg),
            "covariance": covariance_view(X),
        }
    else:
        # MI/ALS-control: emphasize mu/beta and log-variance/covariance structure.
        return {
            "temporal_task": temporal_general_view(X, n_windows=4),
            "spatial_region": spatial_region_view(X, n_regions=4, n_windows=4),
            "frequency_band": frequency_band_view(X, sfreq, bands=((4,8),(8,12),(12,16),(16,24),(24,35))),
            "logvar_stat": logvar_stat_view(X),
            "covariance": covariance_view(X),
        }

print("View builders are ready: P300 ERP views and MI/ALS mu-beta/log-variance views enabled.")


View builders are ready: P300 ERP views and MI/ALS mu-beta/log-variance views enabled.



def best_threshold_by_macro_f1(y_true, prob_pos):
    """
    Choose decision threshold using validation data only.

    P300 data are imbalanced, so a fixed 0.5 threshold is often suboptimal.
    This function never uses test labels when called correctly.
    """
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).reshape(-1)
    if len(np.unique(y_true)) < 2:
        return 0.5
    candidates = np.linspace(0.05, 0.95, 91)
    scores = []
    for thr in candidates:
        pred = (prob_pos >= thr).astype(int)
        scores.append(f1_score(y_true, pred, labels=[0,1], average="macro", zero_division=0))
    return float(candidates[int(np.argmax(scores))])


def compute_binary_metrics(y_true, prob_pos, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).reshape(-1)
    y_pred = (prob_pos >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "threshold": float(threshold),
        "macro_f1": f1_score(y_true, y_pred, labels=[0, 1], average="macro", zero_division=0),
        "f1_positive": f1_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": float(tn / max(tn + fp, 1)),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, prob_pos)
    except Exception:
        out["roc_auc"] = np.nan
    try:
        out["pr_auc"] = average_precision_score(y_true, prob_pos)
    except Exception:
        out["pr_auc"] = np.nan
    return out


def summarize_results(df):
    metric_cols = ["macro_f1", "f1_positive", "precision", "recall", "specificity", "balanced_acc", "accuracy", "roc_auc", "pr_auc"]
    if len(df) == 0:
        return pd.DataFrame()
    summary = df.groupby("model")[metric_cols].agg(["mean", "std", "count"])
    return summary.sort_values(("macro_f1", "mean"), ascending=False)


In [22]:
def choose_validation_subject(train_subjects, seed=42):
    rng = np.random.default_rng(seed)
    train_subjects = np.array(sorted(np.unique(train_subjects).tolist(), key=lambda x: str(x)), dtype=object)
    if len(train_subjects) <= 1:
        return train_subjects[0]
    return rng.choice(train_subjects, size=1)[0]

def _subject_class_sets(subjects, y):
    out = {}
    for s in np.unique(subjects):
        out[s] = set(np.unique(y[subjects == s]).astype(int).tolist())
    return out

def _class_counts_str(y):
    vals, counts = np.unique(np.asarray(y).astype(int), return_counts=True)
    return {int(v): int(c) for v, c in zip(vals, counts)}

def _has_min_binary_counts(y, min_per_class=1):
    vals, counts = np.unique(np.asarray(y).astype(int), return_counts=True)
    if len(vals) < 2:
        return False
    return min(counts) >= int(min_per_class)

def _valid_binary_split(y_train, y_val, y_test, strict=True, min_test_per_class=1, min_val_per_class=1):
    if not strict:
        return True
    return (_has_min_binary_counts(y_train, 1) and
            _has_min_binary_counts(y_val, min_val_per_class) and
            _has_min_binary_counts(y_test, min_test_per_class))

def make_subject_aware_folds(subjects, y, max_outer_folds=None, seed=42, task_type='p300', strict_binary=True):
    subjects = np.asarray(subjects, dtype=object)
    y = np.asarray(y).astype(int)
    rng = np.random.default_rng(seed)
    subj_sets = _subject_class_sets(subjects, y)
    unique_subjects = np.array(sorted(subj_sets.keys(), key=lambda x: str(x)), dtype=object)
    single_class_subjects = all(len(v) == 1 for v in subj_sets.values()) and len(np.unique(y)) == 2
    folds = []
    min_test = getattr(cfg, 'MIN_TEST_PER_CLASS', 1)
    min_val = getattr(cfg, 'MIN_VAL_PER_CLASS', 1)
    if single_class_subjects and getattr(cfg, 'USE_BALANCED_SUBJECT_FOLDS', True):
        class0_subjects = [s for s in unique_subjects if list(subj_sets[s])[0] == 0]
        class1_subjects = [s for s in unique_subjects if list(subj_sets[s])[0] == 1]
        rng.shuffle(class0_subjects); rng.shuffle(class1_subjects)
        n_pairs = min(len(class0_subjects), len(class1_subjects))
        for i in range(n_pairs):
            test_subjects = [class1_subjects[i], class0_subjects[i % len(class0_subjects)]]
            remaining0 = [s for s in class0_subjects if s not in test_subjects]
            remaining1 = [s for s in class1_subjects if s not in test_subjects]
            if len(remaining0) >= 1 and len(remaining1) >= 1:
                val_subjects = [remaining1[i % len(remaining1)], remaining0[i % len(remaining0)]]
            else:
                val_subjects = [choose_validation_subject([s for s in unique_subjects if s not in test_subjects], seed+i)]
            train_subjects = [s for s in unique_subjects if s not in set(test_subjects + val_subjects)]
            train_idx = np.where(np.isin(subjects, train_subjects))[0]
            val_idx = np.where(np.isin(subjects, val_subjects))[0]
            test_idx = np.where(np.isin(subjects, test_subjects))[0]
            if len(train_idx)==0 or len(val_idx)==0 or len(test_idx)==0:
                continue
            if not _valid_binary_split(y[train_idx], y[val_idx], y[test_idx], strict=strict_binary, min_test_per_class=min_test, min_val_per_class=min_val):
                print(f'[SKIP invalid balanced fold] test={test_subjects}, val={val_subjects}, train={_class_counts_str(y[train_idx])}, val={_class_counts_str(y[val_idx])}, test={_class_counts_str(y[test_idx])}')
                continue
            folds.append(('+'.join(map(str,test_subjects)), '+'.join(map(str,val_subjects)), train_idx, val_idx, test_idx))
    else:
        for i, test_s in enumerate(unique_subjects):
            source_subjects = np.unique(subjects[subjects != test_s])
            if len(source_subjects) == 0:
                continue
            val_s = choose_validation_subject(source_subjects, seed+i)
            train_idx = np.where((subjects != test_s) & (subjects != val_s))[0]
            val_idx = np.where(subjects == val_s)[0]
            test_idx = np.where(subjects == test_s)[0]
            if len(train_idx)==0 or len(val_idx)==0 or len(test_idx)==0:
                continue
            if not _valid_binary_split(y[train_idx], y[val_idx], y[test_idx], strict=strict_binary, min_test_per_class=min_test, min_val_per_class=min_val):
                print(f'[SKIP invalid LOSO fold] test={test_s}, val={val_s}, train={_class_counts_str(y[train_idx])}, val={_class_counts_str(y[val_idx])}, test={_class_counts_str(y[test_idx])}')
                continue
            folds.append((str(test_s), str(val_s), train_idx, val_idx, test_idx))
    if max_outer_folds is not None:
        folds = folds[:max_outer_folds]
    print(f'Created {len(folds)} valid subject-level folds | task={task_type} | balanced_subject_folds={single_class_subjects}')
    for j,(ts,vs,tri,vi,tei) in enumerate(folds):
        print(f'  fold {j}: test={ts}, val={vs}, train={_class_counts_str(y[tri])}, val={_class_counts_str(y[vi])}, test={_class_counts_str(y[tei])}')
    if len(folds) == 0:
        raise RuntimeError('No valid folds were created. Check subject IDs and per-subject class counts.')
    return folds

def balance_train_indices_for_task(y, train_idx, task_type, cfg, seed=42):
    """Downsample source-training windows for single-class-subject ALS/control tasks.

    This fixes EEGET's train={0:26880,1:640} problem. It is only applied to tasks listed
    in cfg.BALANCE_TRAIN_INDICES_FOR_TASKS, so BNCI-P300 keeps all non-target trials.
    """
    if str(task_type).lower() not in [str(t).lower() for t in getattr(cfg, 'BALANCE_TRAIN_INDICES_FOR_TASKS', ())]:
        return train_idx
    rng = np.random.default_rng(seed)
    ytr = np.asarray(y)[train_idx].astype(int)
    vals, counts = np.unique(ytr, return_counts=True)
    if len(vals) < 2:
        return train_idx
    min_count = int(counts.min())
    max_per_class = int(max(1, round(min_count * float(getattr(cfg, 'MAX_TRAIN_CLASS_RATIO', 1.0)))))
    selected = []
    for v in vals:
        idx_v = train_idx[ytr == v]
        if len(idx_v) > max_per_class:
            idx_v = rng.choice(idx_v, size=max_per_class, replace=False)
        selected.extend(idx_v.tolist())
    selected = np.asarray(sorted(selected), dtype=int)
    print(f'  [train balance] task={task_type}: before={_class_counts_str(ytr)} after={_class_counts_str(np.asarray(y)[selected])}')
    return selected

def standardize_raw_by_train(X, train_idx, val_idx, test_idx):
    train = X[train_idx].astype(np.float32)
    val = X[val_idx].astype(np.float32)
    test = X[test_idx].astype(np.float32)
    mean = train.mean(axis=(0, 2), keepdims=True)
    std = train.std(axis=(0, 2), keepdims=True) + 1e-6
    return (train-mean)/std, (val-mean)/std, (test-mean)/std


In [23]:
def _scale_and_optionally_cap(Xtr, Xval, Xte, cfg, view_name='', cap_dim=None):
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr).astype(np.float32)
    Xval_s = scaler.transform(Xval).astype(np.float32)
    Xte_s = scaler.transform(Xte).astype(np.float32)
    if getattr(cfg, 'ENABLE_VIEW_PCA_CAP', True):
        dim = int(cap_dim if cap_dim is not None else getattr(cfg, 'VIEW_PCA_DIM', 16))
        if Xtr_s.shape[1] > dim and len(Xtr_s) > dim:
            pca = PCA(n_components=min(dim, Xtr_s.shape[1], len(Xtr_s)-1), random_state=getattr(cfg,'SEED',42))
            Xtr_s = pca.fit_transform(Xtr_s).astype(np.float32)
            Xval_s = pca.transform(Xval_s).astype(np.float32)
            Xte_s = pca.transform(Xte_s).astype(np.float32)
            print(f'  [view PCA] {view_name}: capped to {Xtr_s.shape[1]} dims')
    return Xtr_s, Xval_s, Xte_s, scaler

def fit_transform_views_by_fold(views_all, train_idx, val_idx, test_idx, cfg):
    out = {'train': {}, 'val': {}, 'test': {}}
    scalers = {}
    for name, arr in views_all.items():
        Xtr, Xval, Xte, scaler = _scale_and_optionally_cap(arr[train_idx], arr[val_idx], arr[test_idx], cfg, view_name=name)
        out['train'][name] = Xtr; out['val'][name] = Xval; out['test'][name] = Xte
        scalers[name] = scaler
    return out, scalers

def regularized_covariance_features_from_epochs(X, eps=1e-2):
    N,C,T = X.shape
    iu = np.triu_indices(C)
    feats = np.zeros((N, len(iu[0])), dtype=np.float32)
    for i in range(N):
        Xi = X[i] - X[i].mean(axis=1, keepdims=True)
        cov = (Xi @ Xi.T) / max(T-1, 1)
        scale = np.trace(cov) / max(C, 1)
        cov += eps * (scale if np.isfinite(scale) and scale > 0 else 1.0) * np.eye(C)
        cov = cov / (np.trace(cov) + 1e-8)
        # log-diagonal + normalized upper triangle is stable for low-channel EEGET/ET fallback.
        feats[i] = cov[iu]
    return feats.astype(np.float32)

def make_supervised_fold_views(Xtr_raw, Xval_raw, Xte_raw, ytr, task_type, cfg, seed=42):
    out = {}
    if not getattr(cfg, 'ADD_SUPERVISED_SPATIAL_VIEW', True):
        return out
    task = str(task_type).lower()
    # Riemannian/xDAWN task-aware view; fallback to regularized covariance vector if SPD fails.
    try:
        from pyriemann.tangentspace import TangentSpace
        if task == 'p300':
            from pyriemann.estimation import XdawnCovariances
            pipe = Pipeline([('xdawncov', XdawnCovariances(nfilter=min(4, Xtr_raw.shape[1]), estimator='oas')), ('ts', TangentSpace(metric='riemann'))])
            name = 'xdawn_tangent'
        else:
            from pyriemann.estimation import Covariances
            pipe = Pipeline([('cov', Covariances(estimator='oas')), ('ts', TangentSpace(metric='riemann'))])
            name = 'riemannian_tangent'
        tr = pipe.fit_transform(Xtr_raw, ytr); va = pipe.transform(Xval_raw); te = pipe.transform(Xte_raw)
        tr, va, te, _ = _scale_and_optionally_cap(tr, va, te, cfg, view_name=name, cap_dim=getattr(cfg,'SUPERVISED_VIEW_PCA_DIM',16))
        out[name] = (tr, va, te)
    except Exception as e:
        print(f'  supervised Riemannian/xDAWN view failed, using regularized covariance fallback: {repr(e)}')
        name = 'reg_covariance_tangent'
        tr = regularized_covariance_features_from_epochs(Xtr_raw)
        va = regularized_covariance_features_from_epochs(Xval_raw)
        te = regularized_covariance_features_from_epochs(Xte_raw)
        tr, va, te, _ = _scale_and_optionally_cap(tr, va, te, cfg, view_name=name, cap_dim=getattr(cfg,'SUPERVISED_VIEW_PCA_DIM',16))
        out[name] = (tr, va, te)
    if task != 'p300':
        try:
            from mne.decoding import CSP
            ncomp = max(2, min(6, Xtr_raw.shape[1]-1, len(Xtr_raw)-1))
            csp = CSP(n_components=ncomp, reg='ledoit_wolf', log=True, norm_trace=False)
            tr = csp.fit_transform(Xtr_raw, ytr); va = csp.transform(Xval_raw); te = csp.transform(Xte_raw)
            tr, va, te, _ = _scale_and_optionally_cap(tr, va, te, cfg, view_name='csp_logvar', cap_dim=min(ncomp, getattr(cfg,'SUPERVISED_VIEW_PCA_DIM',16)))
            out['csp_logvar'] = (tr, va, te)
        except Exception as e:
            print(f'  CSP fold view failed: {repr(e)}')
    return out

def add_supervised_views_to_fold(views_fold, supervised_views):
    for name, (tr, va, te) in supervised_views.items():
        views_fold['train'][name] = tr.astype(np.float32)
        views_fold['val'][name] = va.astype(np.float32)
        views_fold['test'][name] = te.astype(np.float32)
    return views_fold

print('Leakage-free fold utilities ready: UCL minimum test counts, EEGET balanced training, regularized covariance fallback.')


Leakage-free fold utilities ready: UCL minimum test counts, EEGET balanced training, regularized covariance fallback.


# 8. Metrics


In [24]:

def best_threshold_by_macro_f1(y_true, prob_pos):
    """
    Choose decision threshold using validation data only.

    P300 data are imbalanced, so a fixed 0.5 threshold is often suboptimal.
    This function never uses test labels when called correctly.
    """
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).reshape(-1)
    if len(np.unique(y_true)) < 2:
        return 0.5
    candidates = np.linspace(0.05, 0.95, 91)
    scores = []
    for thr in candidates:
        pred = (prob_pos >= thr).astype(int)
        scores.append(f1_score(y_true, pred, labels=[0,1], average="macro", zero_division=0))
    return float(candidates[int(np.argmax(scores))])


def compute_binary_metrics(y_true, prob_pos, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).reshape(-1)
    y_pred = (prob_pos >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "threshold": float(threshold),
        "macro_f1": f1_score(y_true, y_pred, labels=[0, 1], average="macro", zero_division=0),
        "f1_positive": f1_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": float(tn / max(tn + fp, 1)),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, prob_pos)
    except Exception:
        out["roc_auc"] = np.nan
    try:
        out["pr_auc"] = average_precision_score(y_true, prob_pos)
    except Exception:
        out["pr_auc"] = np.nan
    return out


def summarize_results(df):
    metric_cols = ["macro_f1", "f1_positive", "precision", "recall", "specificity", "balanced_acc", "accuracy", "roc_auc", "pr_auc"]
    if len(df) == 0:
        return pd.DataFrame()
    summary = df.groupby("model")[metric_cols].agg(["mean", "std", "count"])
    return summary.sort_values(("macro_f1", "mean"), ascending=False)


def validation_selection_score_placeholder():
    pass


# 9. Conventional reference models


In [25]:

def run_sklearn_baselines_fold(views_fold, X_train_raw, X_test_raw, y_train, y_test, seed=42, task_type='p300'):
    rows = []
    Xtr_feat = np.concatenate([views_fold['train'][k] for k in views_fold['train'].keys()], axis=1)
    Xte_feat = np.concatenate([views_fold['test'][k] for k in views_fold['test'].keys()], axis=1)
    task = str(task_type).lower()
    try:
        lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
        lda.fit(Xtr_feat, y_train)
        rows.append(('LDA_flat_multiview', lda.predict_proba(Xte_feat)[:, 1]))
    except Exception as e:
        print('LDA baseline failed:', repr(e))
    try:
        lr = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=seed, n_jobs=-1)
        lr.fit(Xtr_feat, y_train)
        rows.append(('LR_flat_multiview', lr.predict_proba(Xte_feat)[:, 1]))
    except Exception as e:
        print('LR baseline failed:', repr(e))

    if task == 'p300':
        try:
            from pyriemann.estimation import XdawnCovariances
            from pyriemann.tangentspace import TangentSpace
            xdawn_lda = Pipeline([('xdawncov', XdawnCovariances(nfilter=min(4, X_train_raw.shape[1]), estimator='oas')), ('ts', TangentSpace(metric='riemann')), ('clf', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))])
            xdawn_lda.fit(X_train_raw, y_train)
            rows.append(('xDAWN_Tangent_LDA', xdawn_lda.predict_proba(X_test_raw)[:, 1]))
        except Exception as e:
            print('xDAWN+LDA baseline failed:', repr(e))
        try:
            from pyriemann.estimation import XdawnCovariances
            from pyriemann.tangentspace import TangentSpace
            pipe = Pipeline([('xdawncov', XdawnCovariances(nfilter=min(4, X_train_raw.shape[1]), estimator='oas')), ('ts', TangentSpace(metric='riemann')), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=seed))])
            pipe.fit(X_train_raw, y_train)
            rows.append(('xDAWN_Riemannian_LR', pipe.predict_proba(X_test_raw)[:, 1]))
        except Exception as e:
            print('xDAWN/Riemannian baseline failed:', repr(e))
    else:
        try:
            from pyriemann.estimation import Covariances
            from pyriemann.tangentspace import TangentSpace
            pipe = Pipeline([('cov', Covariances(estimator='oas')), ('ts', TangentSpace(metric='riemann')), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=seed))])
            pipe.fit(X_train_raw, y_train)
            rows.append(('Riemannian_Tangent_LR', pipe.predict_proba(X_test_raw)[:, 1]))
        except Exception as e:
            print('Riemannian tangent LR failed; using regularized covariance LR fallback:', repr(e))
            try:
                Xtr_cov = regularized_covariance_features_from_epochs(X_train_raw)
                Xte_cov = regularized_covariance_features_from_epochs(X_test_raw)
                pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=3000, class_weight='balanced', random_state=seed))])
                pipe.fit(Xtr_cov, y_train)
                rows.append(('RegCov_LR', pipe.predict_proba(Xte_cov)[:, 1]))
            except Exception as ee:
                print('Regularized covariance LR fallback failed:', repr(ee))
        try:
            from mne.decoding import CSP
            ncomp = max(2, min(6, X_train_raw.shape[1]-1, len(X_train_raw)-1))
            csp_lda = Pipeline([('csp', CSP(n_components=ncomp, reg='ledoit_wolf', log=True, norm_trace=False)), ('clf', LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'))])
            csp_lda.fit(X_train_raw, y_train)
            rows.append(('CSP_LDA', csp_lda.predict_proba(X_test_raw)[:, 1]))
        except Exception as e:
            print('CSP+LDA baseline failed:', repr(e))
    return rows


# 10. Focused training utilities


In [26]:
def make_balanced_sampler(y):
    y = np.asarray(y).astype(int)
    vals, counts = np.unique(y, return_counts=True)
    count_map = {v: c for v, c in zip(vals, counts)}
    weights = np.array([1.0 / count_map[int(label)] for label in y], dtype=np.float32)
    return WeightedRandomSampler(torch.tensor(weights), num_samples=len(weights), replacement=True)


def model_contains_quantum_layer(model):
    return any(m.__class__.__name__ == "TorchLayer" for m in model.modules())


def preferred_device_for_model(model, cfg):
    if cfg.FORCE_QML_CPU and model_contains_quantum_layer(model):
        return torch.device("cpu")
    return torch.device(DEVICE)


def get_model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device(DEVICE)


def get_class_weights(y_train, target_multiplier=1.0, device=None):
    classes = np.array([0, 1], dtype=np.int64)
    try:
        w = compute_class_weight(class_weight="balanced", classes=classes, y=np.asarray(y_train).astype(int))
    except Exception:
        w = np.ones(2)
    w[1] *= float(target_multiplier)
    return torch.tensor(w, dtype=torch.float32, device=device or torch.device(DEVICE))


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def make_task_criterion(y_train, cfg, device=None):
    weights = get_class_weights(
        y_train, cfg.TARGET_CLASS_WEIGHT_MULTIPLIER, device or torch.device(DEVICE)
    )
    if cfg.LOSS_TYPE.lower() == "focal":
        return FocalLoss(weight=weights, gamma=cfg.FOCAL_GAMMA)
    return nn.CrossEntropyLoss(weight=weights)


def validation_selection_score(metrics):
    """Epoch selection uses discrimination first; threshold still comes from validation only."""
    auc = metrics.get("roc_auc", np.nan)
    auc = 0.5 if not np.isfinite(auc) else float(auc)
    return float(metrics["macro_f1"] + 0.05 * auc)


# 11. Dataset wrappers


In [27]:
class RawEEGTorchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X[:, None, :, :], dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y).astype(np.int64), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y).astype(np.int64), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# 12. EEGNet reference


In [28]:

class EEGNetLite(nn.Module):
    def __init__(self, n_channels, n_times, n_classes=2, F1=8, D=2, F2=16, dropout=0.35):
        super().__init__()
        # Kernel length adapted to epoch length; odd and not too large.
        k1 = min(64, max(16, n_times // 4))
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, k1), padding=(0, k1//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1*D, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, kernel_size=(1, 16), padding=(0, 8), groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(F2, n_classes)
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.head(x)


def evaluate_raw_model(model, loader):
    model.eval()
    device = get_model_device(model)
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            logits = model(xb)
            prob = torch.softmax(logits, dim=1)[:, 1]
            probs.append(prob.detach().cpu().numpy()); ys.append(yb.detach().cpu().numpy())
    return np.concatenate(ys), np.concatenate(probs)


def train_eegnet_fold(Xtr, Xval, Xte, ytr, yval, yte, cfg, seed=42):
    seed_everything(seed)
    train_ds = RawEEGTorchDataset(Xtr, ytr)
    val_ds = RawEEGTorchDataset(Xval, yval)
    test_ds = RawEEGTorchDataset(Xte, yte)

    if cfg.USE_BALANCED_SAMPLER:
        sampler = make_balanced_sampler(ytr)
        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, sampler=sampler)
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, shuffle=False)

    model = EEGNetLite(n_channels=Xtr.shape[1], n_times=Xtr.shape[2], dropout=cfg.DROPOUT).to(DEVICE)
    criterion = make_task_criterion(ytr, cfg, device=torch.device(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=max(2, cfg.PATIENCE//3))

    best_state, best_f1, best_thr, patience = None, -1, 0.5, 0
    for epoch in range(cfg.EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

        yv, pv = evaluate_raw_model(model, val_loader)
        thr = best_threshold_by_macro_f1(yv, pv) if cfg.TUNE_THRESHOLD else 0.5
        val_metrics = compute_binary_metrics(yv, pv, threshold=thr)
        f1 = validation_selection_score(val_metrics)
        scheduler.step(f1)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= cfg.PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    yt_train, pt_train = evaluate_raw_model(model, train_loader)
    train_macro_f1 = compute_binary_metrics(yt_train, pt_train, threshold=best_thr)["macro_f1"]
    yt, pt = evaluate_raw_model(model, test_loader)
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return pt, best_thr, param_count, train_macro_f1


# 13. Legacy and projected VQC architectures


In [29]:
def _small_normal(std):
    def init(tensor):
        with torch.no_grad():
            tensor.normal_(mean=0.0, std=float(std))
        return tensor
    return init

def _identity_block_variational_init(std):
    def init(tensor):
        with torch.no_grad():
            tensor.zero_()
            tensor[0].normal_(mean=0.0, std=float(std))
        return tensor
    return init

def _identity_block_scale_init(tensor):
    with torch.no_grad():
        tensor.zero_()
        tensor[0].fill_(1.0)
    return tensor

def _zero_init(tensor):
    with torch.no_grad():
        tensor.zero_()
    return tensor

def make_legacy_vqc_layer(n_qubits=4, n_layers=1):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        for wire in range(n_qubits):
            qml.RY(inputs[..., wire], wires=wire)
            qml.RZ(inputs[..., n_qubits + wire], wires=wire)
        for layer in range(n_layers):
            for wire in range(n_qubits):
                qml.RY(weights[layer, wire, 0], wires=wire)
                qml.RZ(weights[layer, wire, 1], wires=wire)
            for wire in range(n_qubits):
                qml.CNOT(wires=[wire, (wire + 1) % n_qubits])
        return [qml.expval(qml.PauliZ(wire)) for wire in range(n_qubits)]

    layer = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_qubits, 2)})
    layer.output_dim = n_qubits
    return layer

class LegacyQuantumEncoder(nn.Module):
    def __init__(self, in_dim, n_qubits=4, depth=1, dropout=0.20):
        super().__init__()
        h = max(32, 4 * n_qubits)
        self.input_norm = nn.LayerNorm(in_dim)
        self.project = nn.Sequential(
            nn.Linear(in_dim, h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, 2 * n_qubits), nn.Tanh(),
        )
        self.qlayer = make_legacy_vqc_layer(n_qubits, depth)
        self.post = nn.Sequential(
            nn.LayerNorm(n_qubits), nn.Linear(n_qubits, n_qubits),
            nn.GELU(), nn.LayerNorm(n_qubits),
        )

    def forward(self, x):
        device = x.device
        angles = math.pi * self.project(self.input_norm(x))
        z = self.qlayer(angles)
        if isinstance(z, (list, tuple)):
            z = torch.stack(list(z), dim=-1)
        if z.dim() == 1:
            z = z.unsqueeze(0)
        return self.post(z.to(device=device, dtype=x.dtype))

class LegacySingleVQC(nn.Module):
    def __init__(self, in_dim, n_qubits=4, depth=1, dropout=0.20):
        super().__init__()
        self.encoder = LegacyQuantumEncoder(in_dim, n_qubits, depth, dropout)
        self.classifier = nn.Linear(n_qubits, 2)
    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z


In [30]:
def make_projected_interaction_vqc_layer(n_qubits=4, n_blocks=2, init_std=0.02):
    """VQC-only decision path with repeated projected angles and trainable XX/YY/ZZ coupling.

    Block 1 starts near identity but receives the projected angles. Later blocks start as exact
    identities (zero encoding scales and zero gate parameters) and are activated layerwise.
    """
    if n_qubits < 2:
        raise ValueError("At least two qubits are required for correlation readout.")
    if n_blocks < 2:
        raise ValueError("Layerwise candidate requires at least two quantum blocks.")

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, variational, interaction, input_scales, input_biases):
        for block in range(n_blocks):
            for wire in range(n_qubits):
                ry = input_scales[block, 0, wire] * inputs[..., wire] + input_biases[block, 0, wire]
                rz = input_scales[block, 1, wire] * inputs[..., n_qubits + wire] + input_biases[block, 1, wire]
                qml.RY(ry, wires=wire)
                qml.RZ(rz, wires=wire)

            for wire in range(n_qubits):
                qml.Rot(
                    variational[block, wire, 0],
                    variational[block, wire, 1],
                    variational[block, wire, 2],
                    wires=wire,
                )

            # Alternate the ring ordering so later blocks expose new pairwise interactions.
            order = list(range(n_qubits))
            if block % 2 == 1:
                order = order[::2] + order[1::2]
            for edge in range(n_qubits):
                a, b = order[edge], order[(edge + 1) % n_qubits]
                qml.IsingXX(interaction[block, edge, 0], wires=[a, b])
                qml.IsingYY(interaction[block, edge, 1], wires=[a, b])
                qml.IsingZZ(interaction[block, edge, 2], wires=[a, b])

        measurements = [qml.expval(qml.PauliZ(wire)) for wire in range(n_qubits)]
        measurements += [
            qml.expval(qml.PauliZ(wire) @ qml.PauliZ((wire + 1) % n_qubits))
            for wire in range(n_qubits)
        ]
        return measurements

    shapes = {
        "variational": (n_blocks, n_qubits, 3),
        "interaction": (n_blocks, n_qubits, 3),
        "input_scales": (n_blocks, 2, n_qubits),
        "input_biases": (n_blocks, 2, n_qubits),
    }
    init_method = {
        "variational": _identity_block_variational_init(init_std),
        "interaction": _identity_block_variational_init(init_std),
        "input_scales": _identity_block_scale_init,
        "input_biases": _zero_init,
    }
    layer = qml.qnn.TorchLayer(circuit, shapes, init_method=init_method)
    layer.output_dim = 2 * n_qubits
    layer.n_blocks = n_blocks
    return layer

class ProjectedInteractionQuantumEncoder(nn.Module):
    """Nonlinear angle projector -> VQC -> affine quantum measurement calibration.

    There is deliberately no classical bypass and no post-quantum hidden MLP.
    """
    def __init__(self, in_dim, n_qubits=4, hidden=32, n_blocks=2,
                 dropout=0.10, init_std=0.02):
        super().__init__()
        self.input_norm = nn.LayerNorm(in_dim)
        self.project = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 2 * n_qubits), nn.Tanh(),
        )
        self.qlayer = make_projected_interaction_vqc_layer(n_qubits, n_blocks, init_std)
        self.measure_scale = nn.Parameter(torch.ones(2 * n_qubits))
        self.measure_bias = nn.Parameter(torch.zeros(2 * n_qubits))
        self.dropout = nn.Dropout(dropout * 0.5)

    def forward(self, x):
        out_device = x.device
        angles = math.pi * self.project(self.input_norm(x))
        z = self.qlayer(angles)
        if isinstance(z, (list, tuple)):
            z = torch.stack(list(z), dim=-1)
        if z.dim() == 1:
            z = z.unsqueeze(0)
        z = z.to(device=out_device, dtype=x.dtype)
        z = z * self.measure_scale + self.measure_bias
        return self.dropout(z)

class ProjectedLayerwiseVQC(nn.Module):
    def __init__(self, in_dim, n_qubits=4, hidden=32, n_blocks=2,
                 dropout=0.10, init_std=0.02):
        super().__init__()
        self.encoder = ProjectedInteractionQuantumEncoder(
            in_dim, n_qubits, hidden, n_blocks, dropout, init_std
        )
        self.classifier = nn.Linear(2 * n_qubits, 2)

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z

    def projection_parameters(self):
        return list(self.encoder.input_norm.parameters()) + list(self.encoder.project.parameters())

    def quantum_parameters(self):
        return list(self.encoder.qlayer.parameters())

    def readout_parameters(self):
        return [self.encoder.measure_scale, self.encoder.measure_bias] + list(self.classifier.parameters())


def make_blockwise_reupload_vqc_layer(n_qubits=6, n_blocks=2, init_std=0.02):
    """VQC with a distinct data angle pair for each qubit and each quantum block.

    Input order for every block is:
        [RY angles for all qubits, RZ angles for all qubits]

    The first block receives data at initialization. Later blocks start as identities and
    are activated by the existing layerwise training schedule.
    """
    if n_qubits < 2:
        raise ValueError("At least two qubits are required.")
    if n_blocks < 2:
        raise ValueError("Block-specific re-uploading requires at least two blocks.")

    dev = qml.device("default.qubit", wires=n_qubits)
    angles_per_block = 2 * n_qubits

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, variational, interaction, input_scales, input_biases):
        for block in range(n_blocks):
            offset = block * angles_per_block
            for wire in range(n_qubits):
                raw_ry = inputs[..., offset + wire]
                raw_rz = inputs[..., offset + n_qubits + wire]
                ry = input_scales[block, 0, wire] * raw_ry + input_biases[block, 0, wire]
                rz = input_scales[block, 1, wire] * raw_rz + input_biases[block, 1, wire]
                qml.RY(ry, wires=wire)
                qml.RZ(rz, wires=wire)

            for wire in range(n_qubits):
                qml.Rot(
                    variational[block, wire, 0],
                    variational[block, wire, 1],
                    variational[block, wire, 2],
                    wires=wire,
                )

            order = list(range(n_qubits))
            if block % 2 == 1:
                order = order[::2] + order[1::2]
            for edge in range(n_qubits):
                a, b = order[edge], order[(edge + 1) % n_qubits]
                qml.IsingXX(interaction[block, edge, 0], wires=[a, b])
                qml.IsingYY(interaction[block, edge, 1], wires=[a, b])
                qml.IsingZZ(interaction[block, edge, 2], wires=[a, b])

        measurements = [qml.expval(qml.PauliZ(wire)) for wire in range(n_qubits)]
        measurements += [
            qml.expval(qml.PauliZ(wire) @ qml.PauliZ((wire + 1) % n_qubits))
            for wire in range(n_qubits)
        ]
        return measurements

    shapes = {
        "variational": (n_blocks, n_qubits, 3),
        "interaction": (n_blocks, n_qubits, 3),
        "input_scales": (n_blocks, 2, n_qubits),
        "input_biases": (n_blocks, 2, n_qubits),
    }
    init_method = {
        "variational": _identity_block_variational_init(init_std),
        "interaction": _identity_block_variational_init(init_std),
        "input_scales": _identity_block_scale_init,
        "input_biases": _zero_init,
    }
    layer = qml.qnn.TorchLayer(circuit, shapes, init_method=init_method)
    layer.output_dim = 2 * n_qubits
    layer.n_blocks = n_blocks
    layer.input_dim = 2 * n_qubits * n_blocks
    return layer


## 13.1 Semantic view-to-qubit VQC

Each semantic EEG view group receives a tiny two-layer nonlinear angle projector.  
There is one semantic group per qubit, and no group can bypass the quantum circuit.


In [31]:
SUPERVISED_VIEW_NAMES = {
    "xdawn_tangent", "riemannian_tangent", "csp_logvar", "reg_covariance_tangent"
}

def semantic_group_for_view(view_name):
    name = str(view_name).lower()
    if name in SUPERVISED_VIEW_NAMES:
        return "supervised"
    if name.startswith("temporal"):
        return "temporal"
    if name.startswith("spatial"):
        return "spatial"
    if name.startswith("frequency"):
        return "frequency"
    if name in {"erp_stat", "logvar_stat"} or name.endswith("_stat"):
        return "statistical"
    if "covariance" in name:
        return "covariance"
    raise ValueError(f"Unmapped EEG view {view_name!r}; add an explicit semantic mapping before running.")


def fit_semantic_group_features(views_fold, group_order):
    """Create leakage-free semantic groups without globally mixing the views."""
    group_order = tuple(group_order)
    split_names = ("train", "val", "test")
    for split in split_names:
        if split not in views_fold:
            raise KeyError(f"Missing split: {split}")

    source_map = {group: [] for group in group_order}
    for view_name in views_fold["train"].keys():
        group = semantic_group_for_view(view_name)
        if group not in source_map:
            raise ValueError(f"Group {group!r} is not listed in group_order.")
        source_map[group].append(view_name)

    missing = [group for group, names in source_map.items() if not names]
    if missing:
        raise ValueError(f"Missing required semantic view groups: {missing}")

    grouped = {split: [] for split in split_names}
    group_dims = []
    fitted_scalers = {}
    for group in group_order:
        names = source_map[group]
        arrays = {
            split: np.concatenate([views_fold[split][name] for name in names], axis=1)
            for split in split_names
        }
        scaler = StandardScaler()
        train = scaler.fit_transform(arrays["train"]).astype(np.float32)
        val = scaler.transform(arrays["val"]).astype(np.float32)
        test = scaler.transform(arrays["test"]).astype(np.float32)
        grouped["train"].append(train)
        grouped["val"].append(val)
        grouped["test"].append(test)
        group_dims.append(int(train.shape[1]))
        fitted_scalers[group] = scaler

    return (
        np.concatenate(grouped["train"], axis=1),
        np.concatenate(grouped["val"], axis=1),
        np.concatenate(grouped["test"], axis=1),
        tuple(group_dims),
        source_map,
        fitted_scalers,
    )


class SemanticViewQuantumEncoder(nn.Module):
    """Six tiny view projectors -> six-qubit VQC -> calibrated quantum measurements."""
    def __init__(self, group_dims, hidden=2, n_blocks=2, dropout=0.05, init_std=0.02):
        super().__init__()
        self.group_dims = tuple(int(d) for d in group_dims)
        self.n_qubits = len(self.group_dims)
        if self.n_qubits < 2:
            raise ValueError("At least two semantic groups/qubits are required.")

        # Non-affine normalization adds no trainable classical parameters.
        self.group_projectors = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(dim, elementwise_affine=False),
                nn.Linear(dim, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 2),
                nn.Tanh(),
            )
            for dim in self.group_dims
        ])
        self.qlayer = make_projected_interaction_vqc_layer(
            n_qubits=self.n_qubits, n_blocks=n_blocks, init_std=init_std
        )
        readout_dim = 2 * self.n_qubits
        self.measure_scale = nn.Parameter(torch.ones(readout_dim))
        self.measure_bias = nn.Parameter(torch.zeros(readout_dim))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        offset = 0
        ry_angles, rz_angles = [], []
        for dim, projector in zip(self.group_dims, self.group_projectors):
            x_group = x[:, offset:offset + dim]
            if x_group.shape[1] != dim:
                raise ValueError("Semantic input slicing failed.")
            pair = math.pi * projector(x_group)
            ry_angles.append(pair[:, 0])
            rz_angles.append(pair[:, 1])
            offset += dim
        if offset != x.shape[1]:
            raise ValueError(f"Expected {offset} semantic inputs; received {x.shape[1]}.")

        angles = torch.stack(ry_angles + rz_angles, dim=1)
        out_device = x.device
        z = self.qlayer(angles)
        if isinstance(z, (list, tuple)):
            z = torch.stack(list(z), dim=-1)
        if z.dim() == 1:
            z = z.unsqueeze(0)
        z = z.to(device=out_device, dtype=x.dtype)
        z = z * self.measure_scale + self.measure_bias
        return self.dropout(z)


class SemanticQubitLayerwiseVQC(nn.Module):
    def __init__(self, group_dims, hidden=2, n_blocks=2, dropout=0.05, init_std=0.02):
        super().__init__()
        self.encoder = SemanticViewQuantumEncoder(
            group_dims=group_dims, hidden=hidden, n_blocks=n_blocks,
            dropout=dropout, init_std=init_std,
        )
        self.classifier = nn.Linear(2 * len(group_dims), 2)

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z

    def projection_parameters(self):
        return list(self.encoder.group_projectors.parameters())

    def quantum_parameters(self):
        return list(self.encoder.qlayer.parameters())

    def readout_parameters(self):
        return [self.encoder.measure_scale, self.encoder.measure_bias] + list(self.classifier.parameters())

class SemanticBlockwiseReuploadEncoder(nn.Module):
    """Tiny semantic projectors -> block-specific angles -> six-qubit VQC.

    Each semantic group produces one RY/RZ pair per quantum block. The classifier receives
    only measured quantum observables.
    """
    def __init__(self, group_dims, hidden=2, n_blocks=2, dropout=0.05, init_std=0.02):
        super().__init__()
        self.group_dims = tuple(int(d) for d in group_dims)
        self.n_qubits = len(self.group_dims)
        self.n_blocks = int(n_blocks)
        if self.n_qubits < 2:
            raise ValueError("At least two semantic groups/qubits are required.")
        if self.n_blocks < 2:
            raise ValueError("At least two quantum blocks are required.")

        self.group_projectors = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(dim, elementwise_affine=False),
                nn.Linear(dim, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 2 * self.n_blocks),
                nn.Tanh(),
            )
            for dim in self.group_dims
        ])
        self.qlayer = make_blockwise_reupload_vqc_layer(
            n_qubits=self.n_qubits,
            n_blocks=self.n_blocks,
            init_std=init_std,
        )
        readout_dim = 2 * self.n_qubits
        self.measure_scale = nn.Parameter(torch.ones(readout_dim))
        self.measure_bias = nn.Parameter(torch.zeros(readout_dim))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        offset = 0
        projected = []
        for dim, projector in zip(self.group_dims, self.group_projectors):
            x_group = x[:, offset:offset + dim]
            if x_group.shape[1] != dim:
                raise ValueError("Semantic input slicing failed.")
            pair_by_block = math.pi * projector(x_group)
            pair_by_block = pair_by_block.view(x.shape[0], self.n_blocks, 2)
            projected.append(pair_by_block)
            offset += dim

        if offset != x.shape[1]:
            raise ValueError(f"Expected {offset} semantic inputs; received {x.shape[1]}.")

        # [batch, group/qubit, block, rotation] -> [batch, block, rotation, qubit]
        angles = torch.stack(projected, dim=1).permute(0, 2, 3, 1).contiguous()
        angles = angles.view(x.shape[0], self.n_blocks * 2 * self.n_qubits)

        out_device = x.device
        z = self.qlayer(angles)
        if isinstance(z, (list, tuple)):
            z = torch.stack(list(z), dim=-1)
        if z.dim() == 1:
            z = z.unsqueeze(0)
        z = z.to(device=out_device, dtype=x.dtype)
        z = z * self.measure_scale + self.measure_bias
        return self.dropout(z)


class SemanticReuploadLayerwiseVQC(nn.Module):
    def __init__(self, group_dims, hidden=2, n_blocks=2, dropout=0.05, init_std=0.02):
        super().__init__()
        self.encoder = SemanticBlockwiseReuploadEncoder(
            group_dims=group_dims,
            hidden=hidden,
            n_blocks=n_blocks,
            dropout=dropout,
            init_std=init_std,
        )
        self.classifier = nn.Linear(2 * len(group_dims), 2)

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z), z

    def projection_parameters(self):
        return list(self.encoder.group_projectors.parameters())

    def quantum_parameters(self):
        return list(self.encoder.qlayer.parameters())

    def readout_parameters(self):
        return [self.encoder.measure_scale, self.encoder.measure_bias] + list(self.classifier.parameters())


# 14. VQC training and layerwise optimization


In [32]:
def fit_fold_pca_features(Xtr, Xval, Xte, n_components, seed):
    n_components = int(min(n_components, Xtr.shape[0], Xtr.shape[1]))
    if n_components < 1:
        raise ValueError("PCA requires at least one component.")
    pca = PCA(n_components=n_components, random_state=seed)
    Xtr_pca = pca.fit_transform(Xtr).astype(np.float32)
    Xval_pca = pca.transform(Xval).astype(np.float32)
    Xte_pca = pca.transform(Xte).astype(np.float32)
    scaler = StandardScaler()
    Xtr_pca = scaler.fit_transform(Xtr_pca).astype(np.float32)
    Xval_pca = scaler.transform(Xval_pca).astype(np.float32)
    Xte_pca = scaler.transform(Xte_pca).astype(np.float32)
    return Xtr_pca, Xval_pca, Xte_pca, pca, scaler



def fit_fold_fullview_features(Xtr, Xval, Xte, clip_z=6.0):
    """Preserve the complete multi-view representation while fitting only on training data.

    Each individual view has already been standardized and optionally PCA-capped inside the
    outer fold. This final scaler equalizes the concatenated view blocks without mixing or
    discarding them. No labels or test statistics are used.
    """
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr).astype(np.float32)
    Xval_s = scaler.transform(Xval).astype(np.float32)
    Xte_s = scaler.transform(Xte).astype(np.float32)
    if clip_z is not None and float(clip_z) > 0:
        lim = float(clip_z)
        Xtr_s = np.clip(Xtr_s, -lim, lim).astype(np.float32)
        Xval_s = np.clip(Xval_s, -lim, lim).astype(np.float32)
        Xte_s = np.clip(Xte_s, -lim, lim).astype(np.float32)
    return Xtr_s, Xval_s, Xte_s, scaler

def evaluate_tab_model(model, loader):
    model.eval()
    device = get_model_device(model)
    ys, probs, embeddings = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, z = model(xb)
            ys.append(yb.cpu().numpy())
            probs.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            embeddings.append(z.cpu().numpy())
    return np.concatenate(ys), np.concatenate(probs), np.concatenate(embeddings)

def _grad_l2(parameters):
    total = 0.0
    for p in parameters:
        if p.grad is not None:
            total += float(torch.sum(p.grad.detach() ** 2).cpu())
    return float(total ** 0.5)

def _apply_quantum_block_gradient_mask(model, phase):
    """Mask TorchLayer gradient slices so blocks are trained progressively."""
    qlayer = model.encoder.qlayer
    for name in ("variational", "interaction", "input_scales", "input_biases"):
        p = getattr(qlayer, name, None)
        if p is None or p.grad is None:
            continue
        if phase == "block1":
            p.grad[1:] = 0
        elif phase == "block2":
            p.grad[:1] = 0
        elif phase != "joint":
            raise ValueError(f"Unknown phase: {phase}")

def _set_projection_trainable(model, enabled):
    for p in model.projection_parameters():
        p.requires_grad_(bool(enabled))

def _make_projected_optimizer(model, cfg):
    groups = []
    proj = [p for p in model.projection_parameters() if p.requires_grad]
    quant = [p for p in model.quantum_parameters() if p.requires_grad]
    readout = [p for p in model.readout_parameters() if p.requires_grad]
    if proj:
        groups.append({"params": proj, "lr": cfg.PROJECTION_LR, "weight_decay": cfg.Q_WEIGHT_DECAY})
    if quant:
        groups.append({"params": quant, "lr": cfg.QUANTUM_LR, "weight_decay": 0.0})
    if readout:
        groups.append({"params": readout, "lr": cfg.READOUT_LR, "weight_decay": cfg.Q_WEIGHT_DECAY})
    return torch.optim.AdamW(groups)

def train_legacy_vqc(Xtr, Xval, Xte, ytr, yval, yte, cfg, seed):
    seed_everything(seed)
    train_ds, val_ds, test_ds = TabularDataset(Xtr, ytr), TabularDataset(Xval, yval), TabularDataset(Xte, yte)
    model = LegacySingleVQC(Xtr.shape[1], cfg.N_QUBITS, cfg.LEGACY_VQC_DEPTH, cfg.DROPOUT)
    device = preferred_device_for_model(model, cfg)
    model = model.to(device)
    batch = cfg.QML_BATCH_SIZE_CPU if device.type == "cpu" else cfg.BATCH_SIZE
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True)
    train_eval = DataLoader(train_ds, batch_size=batch, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch, shuffle=False)
    criterion = make_task_criterion(ytr, cfg, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    best_state, best_score, best_thr, stale = None, -np.inf, 0.5, 0
    for _ in range(cfg.EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        yv, pv, _ = evaluate_tab_model(model, val_loader)
        thr = best_threshold_by_macro_f1(yv, pv) if cfg.TUNE_THRESHOLD else 0.5
        metrics = compute_binary_metrics(yv, pv, thr)
        score = validation_selection_score(metrics)
        if score > best_score + 1e-6:
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_score, best_thr, stale = score, thr, 0
        else:
            stale += 1
            if stale >= cfg.PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    ytr_o, ptr, _ = evaluate_tab_model(model, train_eval)
    _, pte, zte = evaluate_tab_model(model, test_loader)
    train_f1 = compute_binary_metrics(ytr_o, ptr, best_thr)["macro_f1"]
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return pte, zte, best_thr, params, train_f1, pd.DataFrame()


In [33]:
def train_projected_layerwise_vqc(Xtr, Xval, Xte, ytr, yval, yte, cfg, seed, model=None):
    seed_everything(seed)
    train_ds, val_ds, test_ds = TabularDataset(Xtr, ytr), TabularDataset(Xval, yval), TabularDataset(Xte, yte)
    if model is None:
        model = ProjectedLayerwiseVQC(
            in_dim=Xtr.shape[1], n_qubits=cfg.N_QUBITS, hidden=cfg.PROJECTOR_HIDDEN,
            n_blocks=cfg.PROJECTED_VQC_BLOCKS, dropout=cfg.PROJECTOR_DROPOUT,
            init_std=cfg.Q_INIT_STD,
        )
    device = preferred_device_for_model(model, cfg)
    model = model.to(device)
    batch = cfg.QML_BATCH_SIZE_CPU if device.type == "cpu" else cfg.BATCH_SIZE
    train_loader = DataLoader(train_ds, batch_size=batch, shuffle=True)
    train_eval = DataLoader(train_ds, batch_size=batch, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=batch, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch, shuffle=False)

    # Fold training is already explicitly balanced where required; do not stack focal loss,
    # target up-weighting and replacement sampling on this candidate.
    criterion = nn.CrossEntropyLoss()
    phases = list(zip(("block1", "block2", "joint"), cfg.Q_PHASE_EPOCHS))
    best_state, best_score, best_thr = None, -np.inf, 0.5
    history = []

    for phase, phase_epochs in phases:
        _set_projection_trainable(model, phase != "block2")
        optimizer = _make_projected_optimizer(model, cfg)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=max(2, cfg.Q_PHASE_PATIENCE // 2)
        )
        stale = 0
        for phase_epoch in range(int(phase_epochs)):
            model.train()
            losses, gp, gq, gh = [], [], [], []
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad(set_to_none=True)
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                _apply_quantum_block_gradient_mask(model, phase)
                gp.append(_grad_l2(model.projection_parameters()))
                gq.append(_grad_l2(model.quantum_parameters()))
                gh.append(_grad_l2(model.readout_parameters()))
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.Q_GRAD_CLIP)
                optimizer.step()
                losses.append(float(loss.detach().cpu()))

            yv, pv, _ = evaluate_tab_model(model, val_loader)
            thr = best_threshold_by_macro_f1(yv, pv) if cfg.TUNE_THRESHOLD else 0.5
            val_metrics = compute_binary_metrics(yv, pv, thr)
            score = validation_selection_score(val_metrics)
            scheduler.step(score)
            history.append({
                "phase": phase, "phase_epoch": phase_epoch,
                "loss": float(np.mean(losses)), "selection_score": score,
                "projector_grad_l2": float(np.mean(gp)),
                "quantum_grad_l2": float(np.mean(gq)),
                "readout_grad_l2": float(np.mean(gh)),
                **val_metrics,
            })
            if score > best_score + 1e-6:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_score, best_thr, stale = score, thr, 0
            else:
                stale += 1
                if stale >= cfg.Q_PHASE_PATIENCE:
                    break

    _set_projection_trainable(model, True)
    if best_state is not None:
        model.load_state_dict(best_state)
    ytr_o, ptr, _ = evaluate_tab_model(model, train_eval)
    _, pte, zte = evaluate_tab_model(model, test_loader)
    train_f1 = compute_binary_metrics(ytr_o, ptr, best_thr)["macro_f1"]
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return pte, zte, best_thr, params, train_f1, pd.DataFrame(history)


# 15. Focused experiment runner


In [34]:
def count_trainable_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def run_focused_experiments(cfg, X, y, subjects, views_all, sfreq, task_type="p300"):
    all_rows, histories = [], {}
    for seed in cfg.SEEDS:
        print("\n" + "=" * 90)
        print("Seed", seed)
        folds = make_subject_aware_folds(
            subjects, y, cfg.MAX_OUTER_FOLDS, seed, task_type,
            strict_binary=cfg.STRICT_BINARY_TEST_FOLDS,
        )
        for fold_id, (test_subject, val_subject, train_idx0, val_idx, test_idx) in enumerate(folds):
            print("\n" + "-" * 80)
            print(f"Fold {fold_id + 1}/{len(folds)} | test={test_subject} | val={val_subject}")
            train_idx = balance_train_indices_for_task(
                y, train_idx0, task_type, cfg, seed + fold_id
            )
            ytr, yval, yte = y[train_idx], y[val_idx], y[test_idx]
            Xtr_raw, Xval_raw, Xte_raw = standardize_raw_by_train(
                X, train_idx, val_idx, test_idx
            )
            views_fold, _ = fit_transform_views_by_fold(
                views_all, train_idx, val_idx, test_idx, cfg
            )
            supervised = make_supervised_fold_views(
                Xtr_raw, Xval_raw, Xte_raw, ytr, task_type, cfg, seed + fold_id
            )
            views_fold = add_supervised_views_to_fold(views_fold, supervised)

            if cfg.RUN_SKLEARN_REFERENCES:
                # run_sklearn_baselines_fold returns (model_name, probability_vector) tuples.
                # Convert each tuple to the same result-row schema used by the neural models.
                for model_name, prob in run_sklearn_baselines_fold(
                    views_fold, Xtr_raw, Xte_raw, ytr, yte, seed, task_type
                ):
                    metrics = compute_binary_metrics(yte, prob, threshold=0.5)
                    row = {
                        "seed": seed, "fold": fold_id, "test_subject": test_subject,
                        "model": model_name,
                        "trainable_params": np.nan, "train_macro_f1": np.nan,
                        "overfit_gap_macro_f1": np.nan,
                        **metrics,
                    }
                    all_rows.append(row)
                    print(row["model"], row)

            if cfg.RUN_EEGNET_REFERENCE:
                try:
                    prob, thr, params, train_f1 = train_eegnet_fold(
                        Xtr_raw, Xval_raw, Xte_raw, ytr, yval, yte, cfg, seed
                    )
                    metrics = compute_binary_metrics(yte, prob, thr)
                    row = {
                        "seed": seed, "fold": fold_id, "test_subject": test_subject,
                        "model": "EEGNetLite_raw", "trainable_params": params,
                        "train_macro_f1": train_f1,
                        "overfit_gap_macro_f1": train_f1 - metrics["macro_f1"],
                        **metrics,
                    }
                    all_rows.append(row); print(row["model"], row)
                except Exception as exc:
                    print("EEGNet failed:", repr(exc))

            Xtr_concat = np.concatenate(list(views_fold["train"].values()), axis=1)
            Xval_concat = np.concatenate(list(views_fold["val"].values()), axis=1)
            Xte_concat = np.concatenate(list(views_fold["test"].values()), axis=1)

            if cfg.RUN_LEGACY_VQC:
                try:
                    Xtr_q, Xval_q, Xte_q, _, _ = fit_fold_pca_features(
                        Xtr_concat, Xval_concat, Xte_concat, cfg.N_QUBITS, seed
                    )
                    prob, emb, thr, params, train_f1, hist = train_legacy_vqc(
                        Xtr_q, Xval_q, Xte_q, ytr, yval, yte, cfg, seed
                    )
                    metrics = compute_binary_metrics(yte, prob, thr)
                    row = {
                        "seed": seed, "fold": fold_id, "test_subject": test_subject,
                        "model": "PCA_single_VQC", "trainable_params": params,
                        "train_macro_f1": train_f1,
                        "overfit_gap_macro_f1": train_f1 - metrics["macro_f1"],
                        "quantum_input_dim": Xtr_q.shape[1],
                        "quantum_blocks": cfg.LEGACY_VQC_DEPTH,
                        "quantum_readout_dim": cfg.N_QUBITS,
                        **metrics,
                    }
                    all_rows.append(row); print(row["model"], row)
                except Exception as exc:
                    print("Legacy VQC failed:", repr(exc))

            if cfg.RUN_FULLVIEW_LAYERWISE_VQC:
                try:
                    Xtr_full, Xval_full, Xte_full, _ = fit_fold_fullview_features(
                        Xtr_concat, Xval_concat, Xte_concat, cfg.FULLVIEW_CLIP_Z
                    )
                    if Xtr_full.shape[1] <= cfg.MIN_FULLVIEW_INPUT_DIM:
                        raise ValueError(
                            f"Full-view reference expected more than {cfg.MIN_FULLVIEW_INPUT_DIM} inputs; "
                            f"got {Xtr_full.shape[1]}."
                        )
                    prob, emb, thr, params, train_f1, hist = train_projected_layerwise_vqc(
                        Xtr_full, Xval_full, Xte_full, ytr, yval, yte, cfg, seed
                    )
                    metrics = compute_binary_metrics(yte, prob, thr)
                    row = {
                        "seed": seed, "fold": fold_id, "test_subject": test_subject,
                        "model": "FullView_Layerwise_VQC", "trainable_params": params,
                        "train_macro_f1": train_f1,
                        "overfit_gap_macro_f1": train_f1 - metrics["macro_f1"],
                        "quantum_input_dim": Xtr_full.shape[1],
                        "quantum_blocks": cfg.PROJECTED_VQC_BLOCKS,
                        "quantum_readout_dim": 2 * cfg.N_QUBITS,
                        **metrics,
                    }
                    all_rows.append(row); print(row["model"], row)
                    histories[(seed, fold_id, "FullView_Layerwise_VQC")] = hist
                except Exception as exc:
                    print("Full-view layerwise VQC failed:", repr(exc))


            if cfg.RUN_SEMANTIC_QUBIT_VQC:
                try:
                    Xtr_sem, Xval_sem, Xte_sem, group_dims, source_map, _ = fit_semantic_group_features(
                        views_fold, cfg.SEMANTIC_GROUP_ORDER
                    )
                    if len(group_dims) != len(cfg.SEMANTIC_GROUP_ORDER):
                        raise ValueError("Semantic group count does not match configured qubit count.")
                    semantic_model = SemanticQubitLayerwiseVQC(
                        group_dims=group_dims,
                        hidden=cfg.SEMANTIC_PROJECTOR_HIDDEN,
                        n_blocks=cfg.PROJECTED_VQC_BLOCKS,
                        dropout=cfg.SEMANTIC_PROJECTOR_DROPOUT,
                        init_std=cfg.Q_INIT_STD,
                    )
                    prob, emb, thr, params, train_f1, hist = train_projected_layerwise_vqc(
                        Xtr_sem, Xval_sem, Xte_sem, ytr, yval, yte, cfg, seed,
                        model=semantic_model,
                    )
                    metrics = compute_binary_metrics(yte, prob, thr)
                    row = {
                        "seed": seed, "fold": fold_id, "test_subject": test_subject,
                        "model": cfg.STEP2C_REFERENCE, "trainable_params": params,
                        "train_macro_f1": train_f1,
                        "overfit_gap_macro_f1": train_f1 - metrics["macro_f1"],
                        "quantum_input_dim": Xtr_sem.shape[1],
                        "quantum_blocks": cfg.PROJECTED_VQC_BLOCKS,
                        "quantum_readout_dim": 2 * len(group_dims),
                        "semantic_qubits": len(group_dims),
                        "semantic_group_dims": "|".join(map(str, group_dims)),
                        "semantic_source_map": json.dumps(source_map, sort_keys=True),
                        **metrics,
                    }
                    all_rows.append(row); print(row["model"], row)
                    histories[(seed, fold_id, cfg.STEP2C_REFERENCE)] = hist
                except Exception as exc:
                    print("Semantic view-to-qubit VQC failed:", repr(exc))


            if cfg.RUN_SEMANTIC_REUPLOAD_VQC:
                try:
                    Xtr_sem, Xval_sem, Xte_sem, group_dims, source_map, _ = fit_semantic_group_features(
                        views_fold, cfg.SEMANTIC_GROUP_ORDER
                    )
                    if len(group_dims) != len(cfg.SEMANTIC_GROUP_ORDER):
                        raise ValueError("Semantic group count does not match configured qubit count.")

                    reupload_model = SemanticReuploadLayerwiseVQC(
                        group_dims=group_dims,
                        hidden=cfg.SEMANTIC_REUPLOAD_HIDDEN,
                        n_blocks=cfg.PROJECTED_VQC_BLOCKS,
                        dropout=cfg.SEMANTIC_PROJECTOR_DROPOUT,
                        init_std=cfg.Q_INIT_STD,
                    )
                    prob, emb, thr, params, train_f1, hist = train_projected_layerwise_vqc(
                        Xtr_sem, Xval_sem, Xte_sem, ytr, yval, yte, cfg, seed,
                        model=reupload_model,
                    )
                    metrics = compute_binary_metrics(yte, prob, thr)
                    row = {
                        "seed": seed,
                        "fold": fold_id,
                        "test_subject": test_subject,
                        "model": cfg.MAIN_CANDIDATE,
                        "trainable_params": params,
                        "train_macro_f1": train_f1,
                        "overfit_gap_macro_f1": train_f1 - metrics["macro_f1"],
                        "quantum_input_dim": 2 * len(group_dims) * cfg.PROJECTED_VQC_BLOCKS,
                        "quantum_blocks": cfg.PROJECTED_VQC_BLOCKS,
                        "quantum_readout_dim": 2 * len(group_dims),
                        "semantic_qubits": len(group_dims),
                        "semantic_group_dims": "|".join(map(str, group_dims)),
                        "semantic_source_map": json.dumps(source_map, sort_keys=True),
                        "block_specific_reupload": True,
                        "angles_per_semantic_group": 2 * cfg.PROJECTED_VQC_BLOCKS,
                        **metrics,
                    }
                    all_rows.append(row)
                    print(row["model"], row)
                    histories[(seed, fold_id, cfg.MAIN_CANDIDATE)] = hist
                except Exception as exc:
                    print("Block-specific semantic re-upload VQC failed:", repr(exc))


            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return pd.DataFrame(all_rows), histories


# 16. Run matched experiments


In [35]:
def _safe_dataset_name(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_")

all_results, all_histories, overview_rows = [], {}, []
for dataset in advisor_loaded_datasets:
    name = dataset["name"]
    print("\n" + "#" * 100)
    print("Focused VQC experiment:", name)
    print("#" * 100)
    X, y, subjects, sfreq = dataset["X"], dataset["y"], dataset["subjects"], dataset["sfreq"]
    views = build_views(X, sfreq, cfg, task_type=dataset.get("task_type", "p300"))
    result, histories = run_focused_experiments(
        cfg, X, y, subjects, views, sfreq, dataset.get("task_type", "p300")
    )
    if len(result):
        result.insert(0, "dataset", name)
        result.insert(1, "task_type", dataset.get("task_type", "unknown"))
        all_results.append(result)
    for key, value in histories.items():
        all_histories[(name,) + tuple(key)] = value
    overview_rows.append({
        "dataset": name, "task_type": dataset.get("task_type", "unknown"),
        "source_name": dataset.get("source_name", ""), "n_trials": len(y),
        "n_subjects": len(np.unique(subjects)), "shape": str(tuple(X.shape)),
        "sfreq": sfreq, "class_distribution": str(dict(zip(*np.unique(y, return_counts=True)))),
    })

if not all_results:
    raise RuntimeError("No experiment results were produced.")

results_df = pd.concat(all_results, ignore_index=True)
dataset_overview = pd.DataFrame(overview_rows)
metric_cols = ["macro_f1", "f1_positive", "precision", "recall", "specificity", "balanced_acc", "accuracy", "roc_auc", "pr_auc"]
summary_df = results_df.groupby(["dataset", "model"]).agg(
    **{f"{m}_mean": (m, "mean") for m in metric_cols},
    **{f"{m}_std": (m, "std") for m in metric_cols},
    n_folds=("macro_f1", "count"),
    trainable_params_mean=("trainable_params", "mean"),
).reset_index().sort_values(["dataset", "macro_f1_mean"], ascending=[True, False])

display(dataset_overview)
display(summary_df)



####################################################################################################
Focused VQC experiment: BNCI-P300
####################################################################################################

Seed 42
Created 3 valid subject-level folds | task=p300 | balanced_subject_folds=False
  fold 0: test=1, val=2, train={0: 5760, 1: 1152}, val={0: 1440, 1: 288}, test={0: 1440, 1: 288}
  fold 1: test=2, val=4, train={0: 5760, 1: 1152}, val={0: 1440, 1: 288}, test={0: 1440, 1: 288}
  fold 2: test=3, val=5, train={0: 5760, 1: 1152}, val={0: 1440, 1: 288}, test={0: 1440, 1: 288}

--------------------------------------------------------------------------------
Fold 1/3 | test=1 | val=2
  [train balance] task=p300: before={0: 5760, 1: 1152} after={0: 1152, 1: 1152}
  [view PCA] temporal_erp: capped to 16 dims
  [view PCA] spatial_region: capped to 16 dims
  [view PCA] frequency_band: capped to 16 dims
  [view PCA] covariance: capped to 16 dims
  [view PCA] x

,dataset,task_type,source_name,n_trials,n_subjects,shape,sfreq,class_distribution
0,BNCI-P300,p300,BNCI2014_009,10368,6,"(10368, 16, 103)",128,"{np.int64(0): np.int64(8640), np.int64(1): np...."
1,UCL-ALS-EEG,mi,Figshare article 28156016,2540,8,"(2540, 22, 119)",128,"{np.int64(0): np.int64(1270), np.int64(1): np...."


,dataset,model,macro_f1_mean,f1_positive_mean,precision_mean,recall_mean,specificity_mean,balanced_acc_mean,accuracy_mean,roc_auc_mean,...,f1_positive_std,precision_std,recall_std,specificity_std,balanced_acc_std,accuracy_std,roc_auc_std,pr_auc_std,n_folds,trainable_params_mean
5,BNCI-P300,xDAWN_Riemannian_LR,0.699198,0.507417,0.498103,0.560185,0.876620,0.718403,0.823881,0.835770,...,0.165250,0.166450,0.240170,0.092417,0.101972,0.064130,0.080501,0.174651,3,NaN
0,BNCI-P300,EEGNetLite_raw,0.699001,0.512685,0.462649,0.583333,0.860880,0.722106,0.814622,0.820566,...,0.048092,0.068628,0.075196,0.038729,0.033879,0.029030,0.044313,0.065248,3,1082.0
6,BNCI-P300,xDAWN_Tangent_LDA,0.689067,0.490330,0.491736,0.542824,0.874537,0.708681,0.819252,0.831366,...,0.153592,0.152962,0.250490,0.102610,0.096528,0.063671,0.076387,0.164553,3,NaN
4,BNCI-P300,SemanticReupload_Layerwise_VQC,0.668552,0.447399,0.454429,0.442130,0.890972,0.666551,0.816165,0.773798,...,0.076789,0.102909,0.053153,0.029634,0.041302,0.033492,0.056984,0.093496,3,440.0
3,BNCI-P300,SemanticQubit_Layerwise_VQC,0.653136,0.419996,0.425729,0.431713,0.886343,0.659028,0.810571,0.776628,...,0.100826,0.031359,0.182427,0.034263,0.074609,0.006855,0.060185,0.101878,3,404.0
1,BNCI-P300,LDA_flat_multiview,0.637627,0.440884,0.393645,0.619213,0.778241,0.698727,0.751736,0.800272,...,0.085216,0.093408,0.287196,0.150860,0.082914,0.085431,0.054937,0.091767,3,NaN
2,BNCI-P300,LR_flat_multiview,0.629643,0.435476,0.369608,0.640046,0.757870,0.698958,0.738233,0.800883,...,0.064838,0.068744,0.286460,0.142943,0.075493,0.073485,0.052980,0.085249,3,NaN
8,UCL-ALS-EEG,EEGNetLite_raw,0.645561,0.699699,0.621242,0.801453,0.506511,0.653982,0.653982,0.718036,...,0.078142,0.081843,0.069344,0.124467,0.096688,0.096688,0.130068,0.145517,3,1210.0
11,UCL-ALS-EEG,Riemannian_Tangent_LR,0.625698,0.602550,0.636344,0.586310,0.675243,0.630777,0.630777,0.700022,...,0.123801,0.039434,0.201872,0.054998,0.073586,0.073586,0.104000,0.126570,3,NaN
9,UCL-ALS-EEG,LDA_flat_multiview,0.601864,0.601179,0.613192,0.598894,0.611690,0.605292,0.605292,0.653396,...,0.027654,0.045702,0.089587,0.135448,0.028211,0.028211,0.017712,0.010490,3,NaN


# 17. Save diagnostics and enforce acceptance gates


In [36]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
results_df.to_csv(os.path.join(cfg.RESULTS_DIR, "fold_level_results.csv"), index=False)
summary_df.to_csv(os.path.join(cfg.RESULTS_DIR, "summary_by_dataset_model.csv"), index=False)
dataset_overview.to_csv(os.path.join(cfg.RESULTS_DIR, "dataset_overview.csv"), index=False)
with open(os.path.join(cfg.RESULTS_DIR, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2, default=str)

for key, hist in all_histories.items():
    dataset_name, seed, fold, model_name = key
    if len(hist):
        filename = f"history_{_safe_dataset_name(dataset_name)}_seed{seed}_fold{fold}_{model_name}.csv"
        hist.to_csv(os.path.join(cfg.RESULTS_DIR, filename), index=False)

candidate = cfg.MAIN_CANDIDATE
references = [
    cfg.STEP2C_REFERENCE,
    "EEGNetLite_raw",
    "xDAWN_Riemannian_LR",
    "xDAWN_Tangent_LDA",
    "Riemannian_Tangent_LR",
    "LR_flat_multiview",
]
metrics = [
    "macro_f1", "f1_positive", "precision", "recall", "specificity",
    "balanced_acc", "accuracy", "roc_auc", "pr_auc",
]
rows = []
keys = ["dataset", "seed", "fold"]

for reference in references:
    a = results_df[results_df.model == candidate]
    b = results_df[results_df.model == reference]
    merged = a[keys + metrics].merge(
        b[keys + metrics], on=keys, suffixes=("_candidate", "_reference")
    )
    for dataset_scope in list(sorted(merged.dataset.unique())) + ["__overall__"]:
        scoped = merged if dataset_scope == "__overall__" else merged[merged.dataset == dataset_scope]
        for metric in metrics:
            if len(scoped) == 0:
                continue
            delta = scoped[f"{metric}_candidate"] - scoped[f"{metric}_reference"]
            rows.append({
                "dataset": dataset_scope,
                "candidate": candidate,
                "reference": reference,
                "metric": metric,
                "n_matched_folds": len(scoped),
                "candidate_mean": scoped[f"{metric}_candidate"].mean(),
                "reference_mean": scoped[f"{metric}_reference"].mean(),
                "mean_delta": delta.mean(),
                "median_delta": delta.median(),
                "win_rate": (delta > 0).mean(),
            })

validation_df = pd.DataFrame(rows)
validation_df.to_csv(
    os.path.join(cfg.RESULTS_DIR, "step2d_matched_fold_validation.csv"),
    index=False,
)

gates = []
for dataset_name in sorted(results_df.dataset.unique()):
    cand = results_df[
        (results_df.dataset == dataset_name) & (results_df.model == candidate)
    ]
    step2c = results_df[
        (results_df.dataset == dataset_name) & (results_df.model == cfg.STEP2C_REFERENCE)
    ]
    eeg = results_df[
        (results_df.dataset == dataset_name) & (results_df.model == "EEGNetLite_raw")
    ]

    def mean(df, metric):
        return float(df[metric].mean()) if len(df) else np.nan

    improves_step2c = (
        mean(cand, "accuracy") > mean(step2c, "accuracy")
        and mean(cand, "macro_f1") > mean(step2c, "macro_f1")
        and mean(cand, "balanced_acc") > mean(step2c, "balanced_acc")
        and (
            mean(cand, "roc_auc") > mean(step2c, "roc_auc")
            or mean(cand, "pr_auc") > mean(step2c, "pr_auc")
        )
    ) if len(cand) and len(step2c) else False

    beats_eegnet = (
        mean(cand, "macro_f1") > mean(eeg, "macro_f1")
        and mean(cand, "balanced_acc") > mean(eeg, "balanced_acc")
        and mean(cand, "accuracy") >= mean(eeg, "accuracy")
    ) if len(cand) and len(eeg) else False

    gates.append({
        "dataset": dataset_name,
        "candidate": candidate,
        "improves_all_core_metrics_vs_step2c": bool(improves_step2c),
        "beats_eegnet_core_metrics": bool(beats_eegnet),
        "accepted_for_integration": bool(improves_step2c and beats_eegnet),
    })

gate_df = pd.DataFrame(gates)
gate_df.to_csv(
    os.path.join(cfg.RESULTS_DIR, "step2d_acceptance_gate.csv"),
    index=False,
)

display(validation_df)
display(gate_df)
print("No superiority claim is valid unless Step-2D passes on both genuine EEG datasets.")


,dataset,candidate,reference,metric,n_matched_folds,candidate_mean,reference_mean,mean_delta,median_delta,win_rate
0,BNCI-P300,SemanticReupload_Layerwise_VQC,SemanticQubit_Layerwise_VQC,macro_f1,3,0.668552,0.653136,0.015417,0.015331,1.000000
1,BNCI-P300,SemanticReupload_Layerwise_VQC,SemanticQubit_Layerwise_VQC,f1_positive,3,0.447399,0.419996,0.027403,0.022478,1.000000
2,BNCI-P300,SemanticReupload_Layerwise_VQC,SemanticQubit_Layerwise_VQC,precision,3,0.454429,0.425729,0.028700,-0.008303,0.333333
3,BNCI-P300,SemanticReupload_Layerwise_VQC,SemanticQubit_Layerwise_VQC,recall,3,0.442130,0.431713,0.010417,0.052083,0.666667
4,BNCI-P300,SemanticReupload_Layerwise_VQC,SemanticQubit_Layerwise_VQC,specificity,3,0.890972,0.886343,0.004630,-0.020139,0.333333
...,...,...,...,...,...,...,...,...,...,...
130,__overall__,SemanticReupload_Layerwise_VQC,LR_flat_multiview,specificity,6,0.792811,0.772995,0.019816,0.125421,0.666667
131,__overall__,SemanticReupload_Layerwise_VQC,LR_flat_multiview,balanced_acc,6,0.616050,0.653191,-0.037141,-0.044937,0.166667
132,__overall__,SemanticReupload_Layerwise_VQC,LR_flat_multiview,accuracy,6,0.690857,0.672829,0.018029,-0.027894,0.333333
133,__overall__,SemanticReupload_Layerwise_VQC,LR_flat_multiview,roc_auc,6,0.709548,0.721142,-0.011595,-0.029004,0.166667


,dataset,candidate,improves_all_core_metrics_vs_step2c,beats_eegnet_core_metrics,accepted_for_integration
0,BNCI-P300,SemanticReupload_Layerwise_VQC,True,False,False
1,UCL-ALS-EEG,SemanticReupload_Layerwise_VQC,False,False,False


No superiority claim is valid unless Step-2D passes on both genuine EEG datasets.


# 18. Interpretation checklist

1. `SemanticQubit_Layerwise_VQC` is the Step-2C matched ablation: one angle pair per semantic group, reused across blocks.
2. `SemanticReupload_Layerwise_VQC` is Step-2D: one distinct angle pair per semantic group **per quantum block**.
3. The circuit, layerwise optimizer, measurements, folds, views, and validation rules remain unchanged.
4. No candidate has a classical prediction anchor, probability fusion, residual bypass, or direct input-to-output path.
5. A Step-2D gain isolates the value of block-specific data re-uploading rather than extra circuit depth.
6. Inspect projector/quantum gradient norms and the selected phase in every history CSV.
7. Reject Step-2D unless `step2d_acceptance_gate.csv` passes on both genuine EEG datasets.
8. Quick mode is only a screen; paper claims require all subject folds and multiple seeds.


# 19. Download result package


In [37]:
import shutil
from google.colab import files
archive = shutil.make_archive(cfg.RESULTS_DIR, "zip", cfg.RESULTS_DIR)
print("Created:", archive)
files.download(archive)


Created: /content/mvqan_vqc_step2d_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>